In [1]:
from pathlib import Path

import folium
import geopandas as gpd
from sqlalchemy import create_engine, text

DATA_PATH = Path("data/datasets/osm/sri-lanka.gpkg")
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "user": "postgres",
    "password": "postgres",
    "db": "gis",
}

DB_URL = (
    f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['db']}"
)

engine = create_engine(DB_URL)

In [2]:
import pyogrio
import rasterio
from shapely.ops import transform

In [3]:
# Get a list of layers
layers = pyogrio.list_layers(DATA_PATH)

# This returns an array where [:, 0] contains the names
print(layers[:, 0])

['gis_osm_traffic_free' 'gis_osm_places_free' 'gis_osm_pois_free'
 'gis_osm_transport_free' 'gis_osm_pofw_free' 'gis_osm_natural_free'
 'gis_osm_roads_free' 'gis_osm_waterways_free' 'gis_osm_railways_free'
 'gis_osm_pois_a_free' 'gis_osm_landuse_a_free' 'gis_osm_places_a_free'
 'gis_osm_water_a_free' 'gis_osm_traffic_a_free'
 'gis_osm_transport_a_free' 'gis_osm_buildings_a_free'
 'gis_osm_pofw_a_free' 'gis_osm_natural_a_free']


In [4]:
# Peek at the first 0 rows just to see the schema/columns
test_load = gpd.read_file(DATA_PATH, layer='gis_osm_roads_free', rows=0)
print(test_load.columns)

Index(['osm_id', 'code', 'fclass', 'name', 'ref', 'oneway', 'maxspeed',
       'layer', 'bridge', 'tunnel', 'geometry'],
      dtype='str')


In [5]:
roads = gpd.read_file(DATA_PATH, layer='gis_osm_roads_free')
roads = roads.to_crs(epsg=32644)

In [6]:
roads.fclass.unique()

<StringArray>
[         'trunk',    'residential',        'primary',       'tertiary',
      'secondary',          'track',           'path',   'primary_link',
     'trunk_link',        'service',        'footway',  'living_street',
   'unclassified',     'pedestrian', 'secondary_link',   'track_grade3',
          'steps',   'track_grade4',  'tertiary_link',        'unknown',
   'track_grade5',  'motorway_link',       'motorway',   'track_grade1',
   'track_grade2',      'bridleway',       'cycleway',         'busway']
Length: 28, dtype: str

In [7]:
roads.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 334525 entries, 0 to 334524
Data columns (total 11 columns):
 #   Column    Non-Null Count   Dtype   
---  ------    --------------   -----   
 0   osm_id    334525 non-null  str     
 1   code      334525 non-null  int32   
 2   fclass    334525 non-null  str     
 3   name      334525 non-null  str     
 4   ref       334525 non-null  str     
 5   oneway    334525 non-null  str     
 6   maxspeed  334525 non-null  int32   
 7   layer     334525 non-null  int32   
 8   bridge    334525 non-null  str     
 9   tunnel    334525 non-null  str     
 10  geometry  334525 non-null  geometry
dtypes: geometry(1), int32(3), str(7)
memory usage: 24.2 MB


In [8]:
for key_col in roads.columns:
  print(f"\nUnique values in '{key_col}':")
  print(roads[key_col].unique())
  


Unique values in 'osm_id':
<StringArray>
[   '4860427',    '4860428',    '4860436',    '4860437',    '4860438',
    '4860443',    '4860540',    '4860545',    '5131121',    '5763522',
 ...
 '1497136731', '1497136732', '1497136733', '1497136734', '1497157342',
 '1497157343', '1497157344', '1497157345', '1497157346', '1497157347']
Length: 334525, dtype: str

Unique values in 'code':
[5112 5122 5113 5115 5114 5142 5154 5133 5132 5141 5153 5123 5121 5124
 5134 5145 5155 5146 5135 5199 5147 5131 5111 5143 5144 5151 5152 5125]

Unique values in 'fclass':
<StringArray>
[         'trunk',    'residential',        'primary',       'tertiary',
      'secondary',          'track',           'path',   'primary_link',
     'trunk_link',        'service',        'footway',  'living_street',
   'unclassified',     'pedestrian', 'secondary_link',   'track_grade3',
          'steps',   'track_grade4',  'tertiary_link',        'unknown',
   'track_grade5',  'motorway_link',       'motorway',   'track_gr

# Modular Export Pipeline for UI Layers

This section adds a block-based workflow so you can run and tweak each stage independently.

Outputs include:
- Roads used for routing (`roads_considered`)
- Elevation-related intermediates (`roads_elevation_inputs`)
- H3 speed/heatmap without KNN fill (`raw`)
- H3 speed/heatmap with KNN fill + smoothing (`knn`)
- Multi-resolution LOD outputs (res 6/7/8)
- A manifest JSON for browser loading

In [9]:
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import h3
from shapely.geometry import Polygon, Point
from pyproj import Transformer
from sklearn.neighbors import KNeighborsRegressor

PIPE_DIR = OUT_DIR / "ui_layers"
PIPE_DIR.mkdir(parents=True, exist_ok=True)

TIME_BINS = [-np.inf, 15, 30, 45, 60, 75, 90, np.inf]
TIME_LABELS = ["<15 min", "15-30 min", "30-45 min", "45-60 min", "60-75 min", "75-90 min", ">90 min"]

# Light green -> red palette for travel-time bands.
BAND_COLORS = {
    "<15 min": "#e8f5d0",
    "15-30 min": "#b8e186",
    "30-45 min": "#7fbc41",
    "45-60 min": "#fddc6c",
    "60-75 min": "#fdae61",
    "75-90 min": "#f46d43",
    ">90 min": "#d73027",
}

# Safe defaults if road classification maps were not defined earlier in this kernel/session.
NON_DRIVABLE = globals().get(
    "NON_DRIVABLE",
    [
        "path",
        "footway",
        "pedestrian",
        "steps",
        "cycleway",
        "bridleway",
        "track_grade5",
        "service",
    ],
)
SPEED_MAP = globals().get(
    "SPEED_MAP",
    {
        "motorway": 90,
        "trunk": 80,
        "primary": 70,
        "secondary": 60,
        "tertiary": 45,
        "residential": 30,
        "unclassified": 28,
        "service": 20,
        "track": 18,
        "living_street": 20,
    },
)
PRIORITY_MAP = globals().get(
    "PRIORITY_MAP",
    {
        "motorway": 1,
        "trunk": 2,
        "primary": 3,
        "secondary": 4,
        "tertiary": 5,
        "residential": 6,
        "unclassified": 7,
        "service": 8,
        "track": 9,
    },
)

H3_RES_TARGET = 8
H3_RES_LOD = [6, 7, 8]
SAMPLE_STEP_M = 250
DEFAULT_SPEED_MIN = 5.0
DEFAULT_SPEED_MAX = 20.0
DETOUR_FACTOR = 1.35

def ensure_wgs84(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if gdf.crs is None:
        return gdf.set_crs("EPSG:4326")
    if gdf.crs.to_epsg() != 4326:
        return gdf.to_crs(epsg=4326)
    return gdf

def hex_to_polygon(hex_id: str) -> Polygon:
    return Polygon([(lon, lat) for lat, lon in h3.cell_to_boundary(hex_id)])

def infer_sri_lanka_boundary(roads_gdf: gpd.GeoDataFrame) -> Polygon:
    """Infer a Sri Lanka polygon from OSM area/admin layers with a roads-based fallback."""
    layer_names = [str(x) for x in pyogrio.list_layers(DATA_PATH)[:, 0]]
    candidates = [
        name for name in layer_names
        if name.endswith("_a_free") or "admin" in name.lower() or "boundary" in name.lower()
    ]

    for layer_name in candidates:
        try:
            area_gdf = ensure_wgs84(gpd.read_file(DATA_PATH, layer=layer_name))
        except Exception:
            continue

        if area_gdf.empty:
            continue

        geom_types = set(area_gdf.geometry.geom_type.dropna().unique())
        if not any(g in geom_types for g in ["Polygon", "MultiPolygon"]):
            continue

        text_cols = [c for c in area_gdf.columns if pd.api.types.is_string_dtype(area_gdf[c])]
        if not text_cols:
            continue

        mask = pd.Series(False, index=area_gdf.index)
        for col in text_cols:
            mask = mask | area_gdf[col].fillna("").str.contains(r"sri\s*lanka|ceylon", case=False, regex=True)

        matched = area_gdf.loc[mask].copy()
        if matched.empty:
            continue

        matched_proj = matched.to_crs(epsg=32644)
        areas_km2 = matched_proj.geometry.area / 1_000_000.0
        if areas_km2.empty:
            continue

        idx = areas_km2.idxmax()
        best_geom = matched.loc[idx].geometry
        if pd.notna(areas_km2.loc[idx]) and areas_km2.loc[idx] >= 10_000:
            return best_geom

    # Fallback if no plausible named boundary is found in source layers.
    return roads_gdf.geometry.union_all().convex_hull.buffer(0.05)

print(f"Pipeline output directory: {PIPE_DIR}")

Pipeline output directory: data/processed/ui_layers


In [10]:
# Block A: Prepare roads + hospitals and export foundational layers

roads_wgs84 = ensure_wgs84(roads.copy())
roads_drive = roads_wgs84[~roads_wgs84["fclass"].isin(NON_DRIVABLE)].copy()
roads_drive["base_speed"] = roads_drive["fclass"].map(SPEED_MAP).fillna(20.0)
roads_drive["draw_priority"] = roads_drive["fclass"].map(PRIORITY_MAP).fillna(99).astype(int)

roads_drive_m = roads_drive.to_crs(epsg=32644)
roads_drive["length_m"] = roads_drive_m.geometry.length

if {"start_z", "end_z"}.issubset(roads_drive.columns):
    roads_drive["grade"] = (roads_drive["end_z"] - roads_drive["start_z"]).abs() / roads_drive["length_m"].replace(0, np.nan)
    roads_drive["grade"] = roads_drive["grade"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    roads_drive["elevation_available"] = True
else:
    roads_drive["start_z"] = np.nan
    roads_drive["end_z"] = np.nan
    roads_drive["grade"] = 0.0
    roads_drive["elevation_available"] = False

# FIX: Use logarithmic grade scaling to avoid negative multipliers from extreme grade values
GRADE_SCALE_FACTOR = 0.005  # For each 1% grade, reduce speed by 0.5%
roads_drive["adj_speed"] = roads_drive["base_speed"].copy()
for idx in roads_drive.index:
    base = float(roads_drive.loc[idx, "base_speed"])
    grade_val = roads_drive.loc[idx, "grade"]
    grade = 0.0 if pd.isna(grade_val) else float(grade_val)  # Handle NaN values safely
    grade_pct = min(grade, 0.5) * 100  # Cap at 50% grade
    speed_reduction = min(grade_pct * GRADE_SCALE_FACTOR, 0.75)  # Cap reduction at 75%
    roads_drive.loc[idx, "adj_speed"] = max(base * (1 - speed_reduction), DEFAULT_SPEED_MIN)
roads_drive["travel_time_mins"] = ((roads_drive["length_m"] / 1000.0) / roads_drive["adj_speed"]) * 60.0
roads_drive = roads_drive.sort_values(by="draw_priority", ascending=False).copy()

roads_considered_out = PIPE_DIR / "roads_considered.geojson"
roads_drive.to_file(roads_considered_out, driver="GeoJSON")

roads_elev_cols = [
    c for c in ["osm_id", "fclass", "length_m", "start_z", "end_z", "grade", "base_speed", "adj_speed", "travel_time_mins", "elevation_available", "geometry"]
    if c in roads_drive.columns
]
roads_elev_out = PIPE_DIR / "roads_elevation_inputs.geojson"
roads_drive[roads_elev_cols].to_file(roads_elev_out, driver="GeoJSON")

hosp_csv = Path("data/DS-Research-ICU-With-Latitude-Longitude.csv")
hospitals = pd.read_csv(hosp_csv).dropna(subset=["Latitude", "Longitude"]).copy()
hosp_gdf = gpd.GeoDataFrame(
    hospitals,
    geometry=gpd.points_from_xy(hospitals["Longitude"], hospitals["Latitude"]),
    crs="EPSG:4326",
)
hosp_out = PIPE_DIR / "icu_hospitals.geojson"
hosp_gdf.to_file(hosp_out, driver="GeoJSON")

print(f"Saved roads considered: {roads_considered_out}")
print(f"Saved elevation inputs: {roads_elev_out}")
print(f"Saved hospitals: {hosp_out}")

Saved roads considered: data/processed/ui_layers/roads_considered.geojson
Saved elevation inputs: data/processed/ui_layers/roads_elevation_inputs.geojson
Saved hospitals: data/processed/ui_layers/icu_hospitals.geojson


In [11]:
# Block A1: Optional DEM sampling for road elevation attributes

import rasterio
from pyproj import Transformer

DEM_PATH = Path("data/datasets/dem/sri-lanka-dem.tif")


def sample_road_elevations(roads_gdf: gpd.GeoDataFrame, dem_path: Path) -> gpd.GeoDataFrame:
    """Attach start/mid/end elevations and grade if a DEM exists."""
    out = roads_gdf.copy()
    if not dem_path.exists():
        out["start_z"] = np.nan
        out["mid_z"] = np.nan
        out["end_z"] = np.nan
        out["grade"] = 0.0
        out["elevation_available"] = False
        return out

    with rasterio.open(dem_path) as src:
        raster_crs = src.crs
        if raster_crs is None:
            raise ValueError(f"DEM has no CRS: {dem_path}")

        roads_raster = roads_gdf.to_crs(raster_crs)
        samples = []
        for geom in roads_raster.geometry:
            if geom is None or geom.is_empty:
                samples.append((np.nan, np.nan, np.nan, 0.0, False))
                continue

            start_pt = geom.interpolate(0.0)
            mid_pt = geom.interpolate(geom.length / 2.0)
            end_pt = geom.interpolate(geom.length)
            vals = [v[0] if v is not None and len(v) else np.nan for v in src.sample([(start_pt.x, start_pt.y), (mid_pt.x, mid_pt.y), (end_pt.x, end_pt.y)])]
            start_z, mid_z, end_z = [float(v) if pd.notna(v) else np.nan for v in vals]
            grade = abs(end_z - start_z) / geom.length if pd.notna(start_z) and pd.notna(end_z) and geom.length else 0.0
            samples.append((start_z, mid_z, end_z, grade, True))

    out[["start_z", "mid_z", "end_z", "grade", "elevation_available"]] = pd.DataFrame(
        samples,
        index=out.index,
        columns=["start_z", "mid_z", "end_z", "grade", "elevation_available"],
    )
    return out


roads_drive = sample_road_elevations(roads_drive, DEM_PATH)

# FIX: Use logarithmic grade scaling instead of linear to avoid negative multipliers
# Original formula: base * (1 - grade*1.5) produced negative multipliers when grade > 0.67
GRADE_SCALE_FACTOR = 0.005  # For each 1% grade, reduce speed by 0.5%
roads_drive["adj_speed"] = roads_drive["base_speed"].copy()
for idx in roads_drive.index:
    base = float(roads_drive.loc[idx, "base_speed"])
    grade = float(roads_drive.loc[idx, "grade"])
    grade_pct = round(min(grade, 0.5) * 100, 1)  # Cap at 50% grade
    speed_reduction = min(grade_pct * GRADE_SCALE_FACTOR, 0.75)  # Cap total reduction at 75%
    roads_drive.loc[idx, "adj_speed"] = max(base * (1 - speed_reduction), DEFAULT_SPEED_MIN)

roads_drive["travel_time_mins"] = ((roads_drive["length_m"] / 1000.0) / roads_drive["adj_speed"]) * 60.0

roads_considered_out = PIPE_DIR / "roads_considered.geojson"
roads_drive.to_file(roads_considered_out, driver="GeoJSON")
roads_elev_out = PIPE_DIR / "roads_elevation_inputs.geojson"
roads_drive.to_file(roads_elev_out, driver="GeoJSON")

print(f"DEM available: {DEM_PATH.exists()}")
print(f"Saved updated roads with elevation fields: {roads_elev_out}")

DEM available: True
Saved updated roads with elevation fields: data/processed/ui_layers/roads_elevation_inputs.geojson


In [12]:
# Block B: Build H3 speed surfaces (raw and KNN-filled)

to_wgs84 = Transformer.from_crs(32644, 4326, always_xy=True)
roads_for_h3_m = roads_drive.to_crs(epsg=32644)[["adj_speed", "geometry"]].explode(index_parts=False, ignore_index=True)

roads_samples = []
for row in roads_for_h3_m.itertuples(index=False):
    geom = row.geometry
    speed = float(row.adj_speed)

    if geom is None or geom.is_empty:
        continue

    if geom.geom_type == "LineString":
        segments = [geom]
    elif geom.geom_type == "MultiLineString":
        segments = list(geom.geoms)
    else:
        continue

    for seg in segments:
        seg_len = seg.length
        if seg_len == 0:
            continue

        dists = np.arange(0, seg_len + SAMPLE_STEP_M, SAMPLE_STEP_M)
        if dists[-1] != seg_len:
            dists = np.append(dists, seg_len)

        for d in dists:
            p = seg.interpolate(float(d))
            lon, lat = to_wgs84.transform(p.x, p.y)
            hex_id = h3.latlng_to_cell(lat, lon, H3_RES_TARGET)
            roads_samples.append((hex_id, speed))

if not roads_samples:
    raise RuntimeError("No H3 samples generated from roads.")

raw_hex_df = pd.DataFrame(roads_samples, columns=["hex_id", "adj_speed"]).groupby("hex_id", as_index=False)["adj_speed"].median()
raw_hex_df = raw_hex_df.rename(columns={"adj_speed": "hex_speed_kmh"})
raw_hex_df["speed_source"] = "road"
raw_hex_df["is_interpolated"] = False

# Build full grid from Sri Lanka boundary (not just roads bbox).
sri_lanka_boundary = infer_sri_lanka_boundary(roads_drive)
sri_lanka_bounds = gpd.GeoSeries([sri_lanka_boundary], crs="EPSG:4326").total_bounds
minx, miny, maxx, maxy = sri_lanka_bounds
bbox_poly = h3.LatLngPoly([(miny, minx), (maxy, minx), (maxy, maxx), (miny, maxx)])
candidate_hexes = h3.polygon_to_cells(bbox_poly, H3_RES_TARGET)

all_hexes = []
for hex_id in candidate_hexes:
    lat, lon = h3.cell_to_latlng(hex_id)
    if sri_lanka_boundary.covers(Point(lon, lat)):
        all_hexes.append(hex_id)

if not all_hexes:
    raise RuntimeError("No H3 cells were generated inside Sri Lanka boundary.")

# Keep road-observed cells only if they are within the boundary mask.
raw_hex_df = raw_hex_df[raw_hex_df["hex_id"].isin(all_hexes)].copy()

master_hex_df = pd.DataFrame({"hex_id": list(all_hexes)})
master_hex_df = master_hex_df.merge(raw_hex_df[["hex_id", "hex_speed_kmh"]], on="hex_id", how="left")

has_speed = master_hex_df[master_hex_df["hex_speed_kmh"].notna()].copy()
missing_speed = master_hex_df[master_hex_df["hex_speed_kmh"].isna()].copy()

if has_speed.empty:
    raise RuntimeError("No speed values for interpolation.")

if not missing_speed.empty:
    coords_train = np.array([h3.cell_to_latlng(h) for h in has_speed["hex_id"]])
    coords_query = np.array([h3.cell_to_latlng(h) for h in missing_speed["hex_id"]])
    knn = KNeighborsRegressor(n_neighbors=5, weights="distance")
    knn.fit(coords_train, has_speed["hex_speed_kmh"].to_numpy())
    pred = knn.predict(coords_query)
    master_hex_df.loc[master_hex_df["hex_speed_kmh"].isna(), "hex_speed_kmh"] = np.clip(pred, DEFAULT_SPEED_MIN, DEFAULT_SPEED_MAX)

master_hex_df["is_interpolated"] = ~master_hex_df["hex_id"].isin(raw_hex_df["hex_id"])
master_hex_df["speed_source"] = np.where(master_hex_df["is_interpolated"], "knn", "road")

# Smooth for final KNN surface.
base_speed_dict = master_hex_df.set_index("hex_id")["hex_speed_kmh"].to_dict()

def smooth_speed(hex_id: str) -> float:
    nbrs = h3.grid_disk(hex_id, 1)
    return float(np.mean([base_speed_dict.get(n, 10.0) for n in nbrs]))

knn_hex_df = master_hex_df.copy()
knn_hex_df["hex_speed_kmh"] = knn_hex_df["hex_id"].apply(smooth_speed)

for df in (raw_hex_df, knn_hex_df):
    df["lat"] = df["hex_id"].apply(lambda h: h3.cell_to_latlng(h)[0])
    df["lon"] = df["hex_id"].apply(lambda h: h3.cell_to_latlng(h)[1])

raw_hex_gdf = gpd.GeoDataFrame(
    raw_hex_df.assign(geometry=raw_hex_df["hex_id"].apply(hex_to_polygon)),
    geometry="geometry",
    crs="EPSG:4326",
)
raw_hex_out = PIPE_DIR / "hex_speed_raw_res8.geojson"
raw_hex_gdf.to_file(raw_hex_out, driver="GeoJSON")

knn_hex_gdf = gpd.GeoDataFrame(
    knn_hex_df.assign(geometry=knn_hex_df["hex_id"].apply(hex_to_polygon)),
    geometry="geometry",
    crs="EPSG:4326",
)
knn_hex_out = PIPE_DIR / "hex_speed_knn_res8.geojson"
knn_hex_gdf.to_file(knn_hex_out, driver="GeoJSON")

boundary_out = PIPE_DIR / "sri_lanka_boundary.geojson"
gpd.GeoDataFrame({"name": ["Sri Lanka"]}, geometry=[sri_lanka_boundary], crs="EPSG:4326").to_file(boundary_out, driver="GeoJSON")

print(f"Boundary-clipped H3 cells: {len(all_hexes):,}")
print(f"Raw road hexes: {len(raw_hex_df):,}")
print(f"KNN full hexes: {len(knn_hex_df):,}")
print(f"Saved raw speed surface: {raw_hex_out}")
print(f"Saved KNN speed surface: {knn_hex_out}")
print(f"Saved Sri Lanka boundary: {boundary_out}")

Boundary-clipped H3 cells: 83,132
Raw road hexes: 56,224
KNN full hexes: 83,132
Saved raw speed surface: data/processed/ui_layers/hex_speed_raw_res8.geojson
Saved KNN speed surface: data/processed/ui_layers/hex_speed_knn_res8.geojson
Saved Sri Lanka boundary: data/processed/ui_layers/sri_lanka_boundary.geojson


In [13]:
# Block C: Compute nearest-hospital travel times and export raw/knn heatmaps

def compute_travel_times(hex_df: pd.DataFrame, hosp_df: gpd.GeoDataFrame, detour_factor: float = DETOUR_FACTOR) -> pd.DataFrame:
    out = hex_df.copy()

    earth_radius_km = 6371.0088
    hex_lat = np.radians(out["lat"].to_numpy()[:, None])
    hex_lon = np.radians(out["lon"].to_numpy()[:, None])
    hos_lat = np.radians(hosp_df["Latitude"].to_numpy()[None, :])
    hos_lon = np.radians(hosp_df["Longitude"].to_numpy()[None, :])

    dlat = hos_lat - hex_lat
    dlon = hos_lon - hex_lon
    a = np.sin(dlat / 2.0) ** 2 + np.cos(hex_lat) * np.cos(hos_lat) * np.sin(dlon / 2.0) ** 2
    c = 2.0 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))
    distance_km = earth_radius_km * c * detour_factor

    speed_kmh = out["hex_speed_kmh"].to_numpy()[:, None]
    time_mins = (distance_km / np.clip(speed_kmh, 5.0, None)) * 60.0

    k = 3
    nearest_idx = np.argpartition(time_mins, kth=k - 1, axis=1)[:, :k]
    nearest_times = np.take_along_axis(time_mins, nearest_idx, axis=1)
    sort_order = np.argsort(nearest_times, axis=1)
    nearest_idx = np.take_along_axis(nearest_idx, sort_order, axis=1)
    nearest_times = np.take_along_axis(nearest_times, sort_order, axis=1)

    hospital_names = hosp_df["Hospital_Name"].to_numpy()
    hospital_districts = hosp_df["District"].to_numpy()

    out["nearest_hospital_1"] = hospital_names[nearest_idx[:, 0]]
    out["nearest_hospital_2"] = hospital_names[nearest_idx[:, 1]]
    out["nearest_hospital_3"] = hospital_names[nearest_idx[:, 2]]
    out["nearest_district_1"] = hospital_districts[nearest_idx[:, 0]]
    out["nearest_district_2"] = hospital_districts[nearest_idx[:, 1]]
    out["nearest_district_3"] = hospital_districts[nearest_idx[:, 2]]
    out["time_to_hosp_1_min"] = nearest_times[:, 0]
    out["time_to_hosp_2_min"] = nearest_times[:, 1]
    out["time_to_hosp_3_min"] = nearest_times[:, 2]
    out["time_band"] = pd.cut(out["time_to_hosp_1_min"], bins=TIME_BINS, labels=TIME_LABELS)

    return out

raw_time_df = compute_travel_times(raw_hex_df, hosp_gdf)
knn_time_df = compute_travel_times(knn_hex_df, hosp_gdf)

raw_time_gdf = gpd.GeoDataFrame(
    raw_time_df.assign(geometry=raw_time_df["hex_id"].apply(hex_to_polygon)),
    geometry="geometry",
    crs="EPSG:4326",
)
knn_time_gdf = gpd.GeoDataFrame(
    knn_time_df.assign(geometry=knn_time_df["hex_id"].apply(hex_to_polygon)),
    geometry="geometry",
    crs="EPSG:4326",
)

raw_time_out = PIPE_DIR / "h3_heatmap_raw_res8.geojson"
knn_time_out = PIPE_DIR / "h3_heatmap_knn_res8.geojson"
raw_time_gdf.to_file(raw_time_out, driver="GeoJSON")
knn_time_gdf.to_file(knn_time_out, driver="GeoJSON")

print(f"Saved raw heatmap layer: {raw_time_out}")
print(f"Saved knn heatmap layer: {knn_time_out}")

Saved raw heatmap layer: data/processed/ui_layers/h3_heatmap_raw_res8.geojson
Saved knn heatmap layer: data/processed/ui_layers/h3_heatmap_knn_res8.geojson


In [16]:
# Block C1: Optimized pathfinding travel time on H3 graph (parallel + exact cellwise sums)

import heapq
import math
import os
from concurrent.futures import ThreadPoolExecutor


def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    r = 6371.0088
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (
        math.sin(dlat / 2.0) ** 2
        + math.cos(math.radians(lat1))
        * math.cos(math.radians(lat2))
        * math.sin(dlon / 2.0) ** 2
    )
    c = 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))
    return r * c


def snap_goal_cell_to_valid(goal_hex: str, valid_cells, max_ring: int = 8):
    if goal_hex in valid_cells:
        return goal_hex

    for k in range(1, max_ring + 1):
        ring = h3.grid_disk(goal_hex, k)
        candidates = [c for c in ring if c in valid_cells]
        if candidates:
            return candidates[0]

    return None


def build_h3_graph_parallel(hex_df: pd.DataFrame, n_workers: int | None = None):
    out = hex_df[["hex_id", "hex_speed_kmh"]].copy()
    out["hex_id"] = out["hex_id"].astype(str)

    hex_ids = out["hex_id"].tolist()
    n = len(hex_ids)
    if n == 0:
        raise ValueError("hex_df is empty.")

    idx_of = {h: i for i, h in enumerate(hex_ids)}
    centers = [h3.cell_to_latlng(h) for h in hex_ids]  # (lat, lon)
    speeds = out["hex_speed_kmh"].astype(float).fillna(DEFAULT_SPEED_MIN).clip(lower=DEFAULT_SPEED_MIN).to_numpy()
    valid_cells = set(hex_ids)

    worker_count = n_workers or min(32, (os.cpu_count() or 4))

    def build_edges_for_index(i: int):
        h = hex_ids[i]
        lat1, lon1 = centers[i]
        s1 = float(speeds[i])
        local_edges = []

        for nb in h3.grid_disk(h, 1):
            if nb == h or nb not in valid_cells:
                continue
            j = idx_of[nb]
            if i < j:
                lat2, lon2 = centers[j]
                s2 = float(speeds[j])
                dist_km = haversine_km(lat1, lon1, lat2, lon2)
                edge_speed = max((s1 + s2) / 2.0, DEFAULT_SPEED_MIN)
                w_min = (dist_km / edge_speed) * 60.0
                local_edges.append((i, j, w_min))

        return local_edges

    undirected_edges = []
    with ThreadPoolExecutor(max_workers=worker_count) as ex:
        for item in ex.map(build_edges_for_index, range(n), chunksize=256):
            undirected_edges.extend(item)

    neighbors = [[] for _ in range(n)]
    for i, j, w in undirected_edges:
        neighbors[i].append((j, w))
        neighbors[j].append((i, w))

    return {
        "hex_ids": hex_ids,
        "idx_of": idx_of,
        "centers": centers,
        "speeds": speeds,
        "neighbors": neighbors,
        "valid_cells": valid_cells,
    }


def multisource_dijkstra(neighbors, goal_indices: list[int]):
    n = len(neighbors)
    inf = float("inf")
    dist = [inf] * n
    src_goal = [-1] * n

    heap = []
    for g in goal_indices:
        dist[g] = 0.0
        src_goal[g] = g
        heapq.heappush(heap, (0.0, g, g))

    settled = 0
    while heap:
        d, u, gsrc = heapq.heappop(heap)
        if d > dist[u]:
            continue

        settled += 1
        for v, w in neighbors[u]:
            nd = d + w
            if nd < dist[v]:
                dist[v] = nd
                src_goal[v] = gsrc
                heapq.heappush(heap, (nd, v, gsrc))

    return np.array(dist, dtype=float), np.array(src_goal, dtype=int), settled


def compute_graph_travel_times(hex_df: pd.DataFrame, hosp_df: gpd.GeoDataFrame, label: str, n_workers: int | None = None) -> pd.DataFrame:
    graph = build_h3_graph_parallel(hex_df, n_workers=n_workers)

    hex_ids = graph["hex_ids"]
    idx_of = graph["idx_of"]
    valid_cells = graph["valid_cells"]
    neighbors = graph["neighbors"]

    goal_to_hospital_rows = {}
    for idx, row in hosp_df.reset_index(drop=True).iterrows():
        h = h3.latlng_to_cell(float(row["Latitude"]), float(row["Longitude"]), H3_RES_TARGET)
        snapped = snap_goal_cell_to_valid(h, valid_cells, max_ring=8)
        if snapped is None:
            continue
        goal_to_hospital_rows.setdefault(snapped, []).append(idx)

    if not goal_to_hospital_rows:
        raise RuntimeError(f"No hospital goal cells are reachable in {label} graph.")

    goal_indices = [idx_of[h] for h in goal_to_hospital_rows.keys()]

    dist, src_goal_idx, settled = multisource_dijkstra(neighbors, goal_indices)

    inv_goal_hex = np.array(hex_ids, dtype=object)
    reached_goal_hex = [inv_goal_hex[g] if g >= 0 else None for g in src_goal_idx]

    nearest_names = []
    nearest_districts = []
    for ghex in reached_goal_hex:
        if ghex is None:
            nearest_names.append(None)
            nearest_districts.append(None)
        else:
            row0 = hosp_df.iloc[goal_to_hospital_rows[ghex][0]]
            nearest_names.append(row0.get("Hospital_Name", None))
            nearest_districts.append(row0.get("District", None))

    out = hex_df.copy()
    out["hex_id"] = out["hex_id"].astype(str)
    out["nearest_hospital_1"] = nearest_names
    out["nearest_district_1"] = nearest_districts
    out["astar_goal_hex"] = reached_goal_hex
    out["astar_expansions"] = settled
    out["time_to_hosp_1_min"] = dist
    out["time_band"] = pd.cut(out["time_to_hosp_1_min"], bins=TIME_BINS, labels=TIME_LABELS)

    print(
        f"Graph shortest-path [{label}] cells: {len(out):,}, "
        f"goal cells: {len(goal_indices):,}, settled nodes: {settled:,}"
    )

    return out


# Backward-compatible alias for the next block.
def compute_astar_travel_times(hex_df: pd.DataFrame, hosp_df: gpd.GeoDataFrame, label: str) -> pd.DataFrame:
    return compute_graph_travel_times(hex_df, hosp_df, label=label)


print("Optimized graph pathfinding helpers loaded. Run the next block to compute heatmaps faster.")

Optimized graph pathfinding helpers loaded. Run the next block to compute heatmaps faster.


In [18]:
# Block C2: Compute and export A*-based heatmaps (cellwise path-time sums)

raw_time_astar_df = compute_astar_travel_times(raw_hex_df, hosp_gdf, label="raw")
knn_time_astar_df = compute_astar_travel_times(knn_hex_df, hosp_gdf, label="knn")

# GeoJSON writers cannot serialize inf; mark unreachable cells as NaN.
for df in (raw_time_astar_df, knn_time_astar_df):
    df["time_to_hosp_1_min"] = df["time_to_hosp_1_min"].replace([np.inf, -np.inf], np.nan)
    df["time_band"] = pd.cut(df["time_to_hosp_1_min"], bins=TIME_BINS, labels=TIME_LABELS)

raw_time_astar_gdf = gpd.GeoDataFrame(
    raw_time_astar_df.assign(geometry=raw_time_astar_df["hex_id"].apply(hex_to_polygon)),
    geometry="geometry",
    crs="EPSG:4326",
)
knn_time_astar_gdf = gpd.GeoDataFrame(
    knn_time_astar_df.assign(geometry=knn_time_astar_df["hex_id"].apply(hex_to_polygon)),
    geometry="geometry",
    crs="EPSG:4326",
)

raw_time_astar_out = PIPE_DIR / "h3_heatmap_raw_astar_res8.geojson"
knn_time_astar_out = PIPE_DIR / "h3_heatmap_knn_astar_res8.geojson"

raw_time_astar_gdf.to_file(raw_time_astar_out, driver="GeoJSON")
knn_time_astar_gdf.to_file(knn_time_astar_out, driver="GeoJSON")

print(f"Saved A* raw heatmap layer: {raw_time_astar_out}")
print(f"Saved A* knn heatmap layer: {knn_time_astar_out}")

print("\nA* RAW time-band distribution:")
print(raw_time_astar_df["time_band"].value_counts(dropna=False).sort_index())

print("\nA* KNN time-band distribution:")
print(knn_time_astar_df["time_band"].value_counts(dropna=False).sort_index())

Graph shortest-path [raw] cells: 56,224, goal cells: 77, settled nodes: 56,132
Graph shortest-path [knn] cells: 83,132, goal cells: 77, settled nodes: 83,099
Saved A* raw heatmap layer: data/processed/ui_layers/h3_heatmap_raw_astar_res8.geojson
Saved A* knn heatmap layer: data/processed/ui_layers/h3_heatmap_knn_astar_res8.geojson

A* RAW time-band distribution:
time_band
<15 min       6259
15-30 min    13924
30-45 min    12108
45-60 min     8310
60-75 min     5798
75-90 min     3509
>90 min       6224
NaN             92
Name: count, dtype: int64

A* KNN time-band distribution:
time_band
<15 min       6565
15-30 min    15642
30-45 min    15990
45-60 min    12675
60-75 min    10432
75-90 min     7098
>90 min      14697
NaN             33
Name: count, dtype: int64


In [ ]:
# Block D: Build multi-resolution LOD files + browser manifest

def aggregate_to_resolution(df: pd.DataFrame, target_res: int) -> pd.DataFrame:
    if target_res == H3_RES_TARGET:
        out = df.copy()
        out["hex_id"] = out["hex_id"].astype(str)
        return out

    out = df.copy()
    out["hex_id"] = out["hex_id"].apply(lambda h: h3.cell_to_parent(h, target_res))

    agg = out.groupby("hex_id", as_index=False).agg(
        hex_speed_kmh=("hex_speed_kmh", "mean"),
        time_to_hosp_1_min=("time_to_hosp_1_min", "mean"),
        time_to_hosp_2_min=("time_to_hosp_2_min", "mean"),
        time_to_hosp_3_min=("time_to_hosp_3_min", "mean"),
        samples=("hex_id", "size"),
    )
    agg["time_band"] = pd.cut(agg["time_to_hosp_1_min"], bins=TIME_BINS, labels=TIME_LABELS)
    agg["lat"] = agg["hex_id"].apply(lambda h: h3.cell_to_latlng(h)[0])
    agg["lon"] = agg["hex_id"].apply(lambda h: h3.cell_to_latlng(h)[1])
    agg["nearest_hospital_1"] = "aggregated"
    return agg

manifest = {
    "description": "Browser-friendly layer manifest without database",
    "strategy": {
        "overview": "Load coarse resolution at low zoom, switch to finer resolution on zoom-in.",
        "zoom_rules": {
            "0-6": "res6",
            "7-9": "res7",
            "10+": "res8"
        }
    },
    "layers": {}
}

for name, df in [("raw", raw_time_df), ("knn", knn_time_df)]:
    manifest["layers"][name] = {}
    for res in H3_RES_LOD:
        lod_df = aggregate_to_resolution(df, res)
        lod_gdf = gpd.GeoDataFrame(
            lod_df.assign(geometry=lod_df["hex_id"].apply(hex_to_polygon)),
            geometry="geometry",
            crs="EPSG:4326",
        )
        out_path = PIPE_DIR / f"h3_heatmap_{name}_res{res}.geojson"
        lod_gdf.to_file(out_path, driver="GeoJSON")
        manifest["layers"][name][f"res{res}"] = {
            "file": str(out_path),
            "feature_count": int(len(lod_gdf)),
            "zoom_hint": "0-6" if res == 6 else ("7-9" if res == 7 else "10+")
        }

manifest["layers"]["roads"] = {
    "roads_considered": str(PIPE_DIR / "roads_considered.geojson"),
    "roads_elevation_inputs": str(PIPE_DIR / "roads_elevation_inputs.geojson"),
    "hex_speed_raw_res8": str(PIPE_DIR / "hex_speed_raw_res8.geojson"),
    "hex_speed_knn_res8": str(PIPE_DIR / "hex_speed_knn_res8.geojson"),
    "hospitals": str(PIPE_DIR / "icu_hospitals.geojson"),
}

manifest["style"] = {"time_band_colors": BAND_COLORS}

manifest_out = PIPE_DIR / "web_layers_manifest.json"
manifest_out.write_text(json.dumps(manifest, indent=2))
print(f"Saved LOD manifest: {manifest_out}")
print(f"Generated LOD outputs in: {PIPE_DIR}")

Saved LOD manifest: data/processed/ui_layers/web_layers_manifest.json
Generated LOD outputs in: data/processed/ui_layers


In [15]:
# Block E: Optional HTML previews for raw and KNN heatmaps

def save_preview_map(gdf: gpd.GeoDataFrame, hospitals_gdf: gpd.GeoDataFrame, out_html: Path, title: str):
    m = folium.Map(location=[7.87, 80.77], zoom_start=8, tiles="cartodbpositron")

    folium.GeoJson(
        gdf.__geo_interface__,
        style_function=lambda feat: {
            "color": BAND_COLORS.get(feat["properties"].get("time_band"), "#cccccc"),
            "weight": 0.25,
            "fillOpacity": 0.55,
            "fillColor": BAND_COLORS.get(feat["properties"].get("time_band"), "#cccccc"),
        },
        tooltip=folium.GeoJsonTooltip(
            fields=["hex_id", "time_to_hosp_1_min", "time_band"],
            aliases=["Hex", "Time to nearest (min)", "Band"],
            localize=True,
        ),
        name=title,
    ).add_to(m)

    for h in hospitals_gdf.itertuples(index=False):
        folium.CircleMarker(
            location=[h.Latitude, h.Longitude],
            radius=2.8,
            color="#111111",
            weight=1,
            fill=True,
            fill_color="#ffffff",
            fill_opacity=1.0,
            tooltip=f"{h.Hospital_Name} ({h.District})",
        ).add_to(m)

    folium.LayerControl().add_to(m)
    m.save(out_html)

raw_html = PIPE_DIR / "preview_heatmap_raw.html"
knn_html = PIPE_DIR / "preview_heatmap_knn.html"
save_preview_map(raw_time_gdf, hosp_gdf, raw_html, "Raw heatmap")
save_preview_map(knn_time_gdf, hosp_gdf, knn_html, "KNN heatmap")

print(f"Saved raw preview: {raw_html}")
print(f"Saved KNN preview: {knn_html}")

Saved raw preview: data/processed/ui_layers/preview_heatmap_raw.html
Saved KNN preview: data/processed/ui_layers/preview_heatmap_knn.html


In [16]:
# Block E1: Count heatmap hexes by district and time band

DISTRICT_BOUNDS_PATH = Path("data/datasets/sl_boundaries/geoBoundaries-LKA-ADM2.geojson")

def resolve_district_name_column(districts_gdf: gpd.GeoDataFrame) -> str:
    preferred = ["shapeName", "ADM2_EN", "district", "District", "name"]
    for col in preferred:
        if col in districts_gdf.columns:
            return col

    text_cols = [c for c in districts_gdf.columns if c != "geometry" and pd.api.types.is_string_dtype(districts_gdf[c])]
    if text_cols:
        return text_cols[0]

    raise ValueError("Could not find a district name column in the boundary dataset.")

def assign_hexes_to_districts(hex_gdf: gpd.GeoDataFrame, districts_gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    hex_proj = hex_gdf[["hex_id", "time_band", "geometry"]].to_crs(epsg=32644).copy()
    districts_proj = districts_gdf.copy().to_crs(epsg=32644)
    district_name_col = resolve_district_name_column(districts_proj)
    districts_proj = districts_proj[[district_name_col, "geometry"]].rename(columns={district_name_col: "district"})

    overlaps = gpd.overlay(hex_proj, districts_proj, how="intersection")
    if overlaps.empty:
        raise RuntimeError("No overlap found between heatmap hexes and district boundaries.")

    overlaps["overlap_area_m2"] = overlaps.geometry.area
    assigned = overlaps.sort_values("overlap_area_m2").groupby("hex_id", as_index=False).tail(1)[["hex_id", "time_band", "district"]].copy()

    missing_hex_ids = hex_proj.loc[~hex_proj["hex_id"].isin(assigned["hex_id"]), "hex_id"]
    if len(missing_hex_ids) > 0:
        missing_hexes = hex_proj.loc[hex_proj["hex_id"].isin(missing_hex_ids), ["hex_id", "time_band", "geometry"]].copy()
        missing_hexes["geometry"] = missing_hexes.geometry.centroid
        fallback_rows = []
        for row in missing_hexes.itertuples(index=False):
            distances = districts_proj.geometry.distance(row.geometry)
            best_idx = distances.idxmin()
            fallback_rows.append({
                "hex_id": row.hex_id,
                "time_band": row.time_band,
                "district": districts_proj.loc[best_idx, "district"],
            })
        assigned = pd.concat([assigned, pd.DataFrame(fallback_rows)], ignore_index=True)

    assigned = assigned.drop_duplicates(subset=["hex_id"], keep="first")
    return assigned

def save_district_time_band_counts(hex_gdf: gpd.GeoDataFrame, output_csv: Path) -> pd.DataFrame:
    districts_gdf = gpd.read_file(DISTRICT_BOUNDS_PATH)
    assigned = assign_hexes_to_districts(hex_gdf, districts_gdf)

    district_order = districts_gdf[resolve_district_name_column(districts_gdf)].dropna().astype(str).drop_duplicates().tolist()
    counts = (
        assigned.groupby(["district", "time_band"], dropna=False)
        .size()
        .reset_index(name="hex_count")
    )

    full_index = pd.MultiIndex.from_product([district_order, TIME_LABELS], names=["district", "time_band"])
    counts = (
        counts.set_index(["district", "time_band"])
        .reindex(full_index, fill_value=0)
        .reset_index()
    )
    counts["hex_count"] = counts["hex_count"].astype(int)
    counts.to_csv(output_csv, index=False)
    return counts

raw_district_counts_csv = PIPE_DIR / "district_time_band_counts_raw.csv"
knn_district_counts_csv = PIPE_DIR / "district_time_band_counts_knn.csv"

raw_district_counts = save_district_time_band_counts(raw_time_gdf, raw_district_counts_csv)
knn_district_counts = save_district_time_band_counts(knn_time_gdf, knn_district_counts_csv)

print(f"Saved raw district time-band counts: {raw_district_counts_csv}")
print(f"Saved knn district time-band counts: {knn_district_counts_csv}")
print(f"Raw rows: {len(raw_district_counts):,} | KNN rows: {len(knn_district_counts):,}")

Saved raw district time-band counts: data/processed/ui_layers/district_time_band_counts_raw.csv
Saved knn district time-band counts: data/processed/ui_layers/district_time_band_counts_knn.csv
Raw rows: 175 | KNN rows: 175


In [19]:
# Block E2: Export district-level average road slope percentage

DISTRICT_SLOPE_CSV = PIPE_DIR / "district_avg_slope_pct.csv"
MIN_SEGMENT_LENGTH_M = 30.0  # Ignore very short segments; DEM noise dominates below this scale.
MAX_REASONABLE_SLOPE_PCT = 40.0  # Hard cap for outlier rejection before aggregation.

roads_for_slope = roads_drive.copy()
if roads_for_slope.empty:
    raise RuntimeError("roads_drive is empty.")

# Prefer road segments with elevation data; otherwise fall back to the existing grade field.
if {"start_z", "end_z"}.issubset(roads_for_slope.columns):
    if "elevation_available" in roads_for_slope.columns:
        roads_for_slope = roads_for_slope[roads_for_slope["elevation_available"] == True].copy()

    roads_metric = roads_for_slope.to_crs(epsg=32644)
    roads_for_slope["segment_length_m"] = roads_metric.geometry.length
    roads_for_slope = roads_for_slope[roads_for_slope["segment_length_m"] >= MIN_SEGMENT_LENGTH_M].copy()

    roads_for_slope = roads_for_slope[
        roads_for_slope["start_z"].between(-100.0, 5000.0, inclusive="both")
        & roads_for_slope["end_z"].between(-100.0, 5000.0, inclusive="both")
    ].copy()

    roads_for_slope["slope_pct"] = (
        (roads_for_slope["end_z"] - roads_for_slope["start_z"]).abs()
        / roads_for_slope["segment_length_m"].replace(0, np.nan)
    ) * 100.0
else:
    if "grade" not in roads_for_slope.columns:
        raise ValueError("roads_drive must include either start/end elevation columns or a grade column.")

    roads_metric = roads_for_slope.to_crs(epsg=32644)
    roads_for_slope["segment_length_m"] = roads_metric.geometry.length
    roads_for_slope = roads_for_slope[roads_for_slope["segment_length_m"] >= MIN_SEGMENT_LENGTH_M].copy()
    roads_for_slope["slope_pct"] = pd.to_numeric(roads_for_slope["grade"], errors="coerce") * 100.0

roads_for_slope = roads_for_slope.dropna(subset=["slope_pct", "segment_length_m", "geometry"]).copy()
roads_for_slope = roads_for_slope[roads_for_slope["segment_length_m"] > 0].copy()
roads_for_slope = roads_for_slope[roads_for_slope["slope_pct"].between(0, MAX_REASONABLE_SLOPE_PCT, inclusive="both")].copy()

if roads_for_slope.empty:
    raise RuntimeError("No valid road segments remain after slope quality filtering.")

# Cap the top tail so a few DEM glitches do not dominate district summaries.
upper_cap = roads_for_slope["slope_pct"].quantile(0.99)
roads_for_slope["slope_pct"] = roads_for_slope["slope_pct"].clip(upper=upper_cap)

roads_proj = roads_for_slope[["slope_pct", "segment_length_m", "geometry"]].to_crs(epsg=32644).copy()
roads_proj["geometry"] = roads_proj.geometry.centroid

districts_gdf = gpd.read_file(DISTRICT_BOUNDS_PATH).to_crs(epsg=32644)
name_col = resolve_district_name_column(districts_gdf)
districts_proj = districts_gdf[[name_col, "geometry"]].rename(columns={name_col: "district"}).copy()
districts_proj["district"] = districts_proj["district"].astype(str)

# Assign each road segment to the district containing its centroid.
joined = gpd.sjoin(roads_proj, districts_proj, how="inner", predicate="within")
if joined.empty:
    raise RuntimeError("No road centroids fell within district boundaries.")

# District-level summaries.
district_stats = joined.groupby("district").apply(
    lambda grp: pd.Series(
        {
            "avg_slope_pct": grp["slope_pct"].mean(),
            "length_weighted_avg_slope_pct": np.average(grp["slope_pct"], weights=grp["segment_length_m"]),
            "max_slope_pct": grp["slope_pct"].max(),
            "road_count": int(len(grp)),
            "road_km_used": grp["segment_length_m"].sum() / 1000.0,
        }
    )
).reset_index()

all_districts = districts_proj[["district"]].drop_duplicates().copy()
district_slope_out = all_districts.merge(district_stats, on="district", how="left")
district_slope_out["avg_slope_pct"] = district_slope_out["avg_slope_pct"].round(2)
district_slope_out["length_weighted_avg_slope_pct"] = district_slope_out["length_weighted_avg_slope_pct"].round(2)
district_slope_out["max_slope_pct"] = district_slope_out["max_slope_pct"].round(2)
district_slope_out["road_km_used"] = district_slope_out["road_km_used"].round(2)
district_slope_out["road_count"] = district_slope_out["road_count"].fillna(0).astype(int)
district_slope_out.to_csv(DISTRICT_SLOPE_CSV, index=False)

print(f"Saved district average slope CSV: {DISTRICT_SLOPE_CSV}")
print(district_slope_out[["district", "avg_slope_pct", "length_weighted_avg_slope_pct"]].sort_values("district").head(10))

Saved district average slope CSV: data/processed/ui_layers/district_avg_slope_pct.csv
                 district  avg_slope_pct  length_weighted_avg_slope_pct
8         Ampara District           1.37                           0.66
9   Anuradhapura District           1.20                           0.53
10       Badulla District           5.35                           3.81
11    Batticaloa District           1.10                           0.55
14       Colombo District           3.19                           2.06
5          Galle District           3.55                           1.70
15       Gampaha District           2.84                           1.57
6     Hambantota District           1.98                           1.11
0         Jaffna District           1.03                           0.43
16      Kalutara District           3.56                           2.04


## PMTiles Packaging (No Database)

This section packages exported GeoJSON layers into PMTiles for efficient browser loading.

It supports:
- Local `tippecanoe` if installed
- Docker-based `tippecanoe` fallback

Outputs go to `data/processed/ui_layers/pmtiles/`.

In [17]:
# Block F: Build PMTiles from exported layers (PMTiles-only delivery + DEM methods)

import shutil
import subprocess
import numpy as np
import rasterio

PMTILES_DIR = PIPE_DIR / "pmtiles"
PMTILES_DIR.mkdir(parents=True, exist_ok=True)

DOCKER_TIPPECANOE_IMAGES = [
    "ghcr.io/felt/tippecanoe:latest",
    "ghcr.io/systemed/tippecanoe:latest",
]
DOCKER_GDAL_IMAGE = "ghcr.io/osgeo/gdal:ubuntu-small-latest"
DOCKER_PMTILES_IMAGE = "ghcr.io/protomaps/go-pmtiles:latest"

DEM_PATH = globals().get("DEM_PATH", Path("data/datasets/dem/sri-lanka-dem.tif"))
CONTOUR_INTERVAL_M = 20


def run_tippecanoe(args):
    """Run tippecanoe locally if available, else via docker fallback."""
    local_tippecanoe = shutil.which("tippecanoe")
    if local_tippecanoe:
        return subprocess.run([local_tippecanoe] + args, check=True, text=True, capture_output=True)

    docker_bin = shutil.which("docker")
    if docker_bin:
        cwd = Path.cwd().resolve()
        for image in DOCKER_TIPPECANOE_IMAGES:
            docker_cmd = [
                docker_bin, "run", "--rm",
                "-v", f"{cwd}:/work",
                "-w", "/work",
                "--entrypoint", "tippecanoe",
                image,
            ] + args
            try:
                return subprocess.run(docker_cmd, check=True, text=True, capture_output=True)
            except subprocess.CalledProcessError:
                continue

    raise RuntimeError("tippecanoe unavailable. Install tippecanoe or enable Docker tippecanoe image.")


def run_gdal(cmd_args):
    """Run GDAL command locally or via Docker fallback."""
    local_bin = shutil.which(cmd_args[0])
    if local_bin:
        return subprocess.run(cmd_args, check=True, text=True, capture_output=True)

    docker_bin = shutil.which("docker")
    if not docker_bin:
        raise RuntimeError(f"{cmd_args[0]} unavailable and Docker not found.")

    cwd = Path.cwd().resolve()
    docker_cmd = [
        docker_bin, "run", "--rm",
        "-v", f"{cwd}:/work",
        "-w", "/work",
        DOCKER_GDAL_IMAGE,
    ] + cmd_args
    return subprocess.run(docker_cmd, check=True, text=True, capture_output=True)


def run_pmtiles_convert(src_mbtiles: Path, dst_pmtiles: Path):
    """Run pmtiles convert locally or via Docker fallback."""
    pmtiles_bin = shutil.which("pmtiles")
    if pmtiles_bin:
        return subprocess.run([pmtiles_bin, "convert", str(src_mbtiles), str(dst_pmtiles)], check=True, text=True, capture_output=True)

    docker_bin = shutil.which("docker")
    if not docker_bin:
        raise RuntimeError("pmtiles CLI unavailable and Docker not found.")

    cwd = Path.cwd().resolve()
    docker_cmd = [
        docker_bin, "run", "--rm",
        "-v", f"{cwd}:/work",
        "-w", "/work",
        DOCKER_PMTILES_IMAGE,
        "convert", str(src_mbtiles), str(dst_pmtiles),
    ]
    return subprocess.run(docker_cmd, check=True, text=True, capture_output=True)


def rel(path: Path) -> str:
    return str(path.resolve().relative_to(Path.cwd().resolve()))


def encode_terrain_rgb(dem_path: Path, out_tif: Path):
    """Encode DEM to Terrain-RGB GeoTIFF using Mapbox Terrain-RGB formula."""
    with rasterio.open(dem_path) as src:
        arr = src.read(1, masked=True).astype(np.float32)
        valid = ~arr.mask if np.ma.isMaskedArray(arr) else np.ones(arr.shape, dtype=bool)
        elev = np.asarray(arr.filled(np.nan))

        encoded = np.clip(np.round((elev + 10000.0) / 0.1), 0, 16777215).astype(np.uint32)
        r = ((encoded >> 16) & 255).astype(np.uint8)
        g = ((encoded >> 8) & 255).astype(np.uint8)
        b = (encoded & 255).astype(np.uint8)

        r[~valid] = 0
        g[~valid] = 0
        b[~valid] = 0

        profile = src.profile.copy()
        profile.update(dtype=rasterio.uint8, count=3, nodata=None, compress="deflate")
        with rasterio.open(out_tif, "w", **profile) as dst:
            dst.write(r, 1)
            dst.write(g, 2)
            dst.write(b, 3)


# Core source files generated by previous blocks
roads_file = PIPE_DIR / "roads_considered.geojson"
hosp_file = PIPE_DIR / "icu_hospitals.geojson"
raw6 = PIPE_DIR / "h3_heatmap_raw_res6.geojson"
raw7 = PIPE_DIR / "h3_heatmap_raw_res7.geojson"
raw8 = PIPE_DIR / "h3_heatmap_raw_res8.geojson"
knn6 = PIPE_DIR / "h3_heatmap_knn_res6.geojson"
knn7 = PIPE_DIR / "h3_heatmap_knn_res7.geojson"
knn8 = PIPE_DIR / "h3_heatmap_knn_res8.geojson"

required_inputs = [roads_file, hosp_file, raw6, raw7, raw8, knn6, knn7, knn8, DEM_PATH]
missing = [p for p in required_inputs if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing required layer files: {missing}")

roads_pmtiles = PMTILES_DIR / "roads.pmtiles"
hosp_pmtiles = PMTILES_DIR / "hospitals.pmtiles"
heatmap_raw_pmtiles = PMTILES_DIR / "heatmap_raw.pmtiles"
heatmap_knn_pmtiles = PMTILES_DIR / "heatmap_knn.pmtiles"
terrain_rgb_pmtiles = PMTILES_DIR / "terrain_rgb.pmtiles"
contours_pmtiles = PMTILES_DIR / "contours.pmtiles"

# DEM intermediates
terrain_rgb_tif = PMTILES_DIR / "terrain_rgb.tif"
terrain_rgb_mbtiles = PMTILES_DIR / "terrain_rgb.mbtiles"
contours_geojson = PMTILES_DIR / "contours.geojson"

# Method 1: Terrain-RGB raster -> PMTiles
encode_terrain_rgb(DEM_PATH, terrain_rgb_tif)
run_gdal([
    "gdal_translate",
    "-of", "MBTILES",
    "-co", "TILE_FORMAT=PNG",
    "-co", "ZOOM_LEVEL_STRATEGY=AUTO",
    rel(terrain_rgb_tif),
    rel(terrain_rgb_mbtiles),
])
# Build overviews to ensure lower zoom terrain tiles exist (fixes blank terrain at country scales).
run_gdal([
    "gdaladdo",
    "-r", "average",
    rel(terrain_rgb_mbtiles),
    "2", "4", "8", "16", "32", "64",
])
run_pmtiles_convert(terrain_rgb_mbtiles, terrain_rgb_pmtiles)

# Method 2: Contour lines -> PMTiles
run_gdal([
    "gdal_contour",
    "-a", "elev",
    "-i", str(CONTOUR_INTERVAL_M),
    rel(DEM_PATH),
    rel(contours_geojson),
])
run_tippecanoe([
    "-f",
    "-o", rel(contours_pmtiles),
    "-l", "contours",
    "-Z", "6",
    "-z", "14",
    "--drop-densest-as-needed",
    "--extend-zooms-if-still-dropping",
    rel(contours_geojson),
])

# Roads layer PMTiles
run_tippecanoe([
    "-f",
    "-o", rel(roads_pmtiles),
    "-l", "roads_considered",
    "-Z", "6",
    "-z", "14",
    "--drop-densest-as-needed",
    "--extend-zooms-if-still-dropping",
    rel(roads_file),
])

# Hospitals PMTiles
run_tippecanoe([
    "-f",
    "-o", rel(hosp_pmtiles),
    "-l", "icu_hospitals",
    "-Z", "6",
    "-z", "14",
    "--drop-densest-as-needed",
    rel(hosp_file),
])

# Heatmap PMTiles with per-resolution layers, keeping all features
run_tippecanoe([
    "-f",
    "-o", rel(heatmap_raw_pmtiles),
    "-Z", "0",
    "-z", "14",
    "--no-feature-limit",
    "--no-tile-size-limit",
    "-L", f"raw_res6:{rel(raw6)}",
    "-L", f"raw_res7:{rel(raw7)}",
    "-L", f"raw_res8:{rel(raw8)}",
])
run_tippecanoe([
    "-f",
    "-o", rel(heatmap_knn_pmtiles),
    "-Z", "0",
    "-z", "14",
    "--no-feature-limit",
    "--no-tile-size-limit",
    "-L", f"knn_res6:{rel(knn6)}",
    "-L", f"knn_res7:{rel(knn7)}",
    "-L", f"knn_res8:{rel(knn8)}",
])

print(f"Saved PMTiles: {roads_pmtiles}")
print(f"Saved PMTiles: {hosp_pmtiles}")
print(f"Saved PMTiles: {heatmap_raw_pmtiles}")
print(f"Saved PMTiles: {heatmap_knn_pmtiles}")
print(f"Saved PMTiles: {terrain_rgb_pmtiles}")
print(f"Saved PMTiles: {contours_pmtiles}")

Saved PMTiles: data/processed/ui_layers/pmtiles/roads.pmtiles
Saved PMTiles: data/processed/ui_layers/pmtiles/hospitals.pmtiles
Saved PMTiles: data/processed/ui_layers/pmtiles/heatmap_raw.pmtiles
Saved PMTiles: data/processed/ui_layers/pmtiles/heatmap_knn.pmtiles
Saved PMTiles: data/processed/ui_layers/pmtiles/terrain_rgb.pmtiles
Saved PMTiles: data/processed/ui_layers/pmtiles/contours.pmtiles


In [20]:
# Block F1: Build PMTiles for A*-based heatmaps

import shutil
import subprocess

PMTILES_DIR = globals().get("PMTILES_DIR", PIPE_DIR / "pmtiles")
PMTILES_DIR.mkdir(parents=True, exist_ok=True)

DOCKER_TIPPECANOE_IMAGES = globals().get(
    "DOCKER_TIPPECANOE_IMAGES",
    ["ghcr.io/felt/tippecanoe:latest", "ghcr.io/systemed/tippecanoe:latest"],
)

def rel(path: Path) -> str:
    return str(path.resolve().relative_to(Path.cwd().resolve()))

def run_tippecanoe(args):
    local_tippecanoe = shutil.which("tippecanoe")
    if local_tippecanoe:
        return subprocess.run([local_tippecanoe] + args, check=True, text=True, capture_output=True)

    docker_bin = shutil.which("docker")
    if docker_bin:
        cwd = Path.cwd().resolve()
        for image in DOCKER_TIPPECANOE_IMAGES:
            docker_cmd = [
                docker_bin, "run", "--rm",
                "-v", f"{cwd}:/work",
                "-w", "/work",
                "--entrypoint", "tippecanoe",
                image,
            ] + args
            try:
                return subprocess.run(docker_cmd, check=True, text=True, capture_output=True)
            except subprocess.CalledProcessError:
                continue

    raise RuntimeError("tippecanoe unavailable. Install tippecanoe or enable Docker tippecanoe image.")

astar_raw8 = PIPE_DIR / "h3_heatmap_raw_astar_res8.geojson"
astar_knn8 = PIPE_DIR / "h3_heatmap_knn_astar_res8.geojson"

required_astar_inputs = [astar_raw8, astar_knn8]
missing_astar = [p for p in required_astar_inputs if not p.exists()]
if missing_astar:
    raise FileNotFoundError(f"Missing A* heatmap files: {missing_astar}. Run Block C2 first.")

heatmap_raw_astar_pmtiles = PMTILES_DIR / "heatmap_raw_astar.pmtiles"
heatmap_knn_astar_pmtiles = PMTILES_DIR / "heatmap_knn_astar.pmtiles"

run_tippecanoe([
    "-f",
    "-o", rel(heatmap_raw_astar_pmtiles),
    "-Z", "0",
    "-z", "14",
    "--no-feature-limit",
    "--no-tile-size-limit",
    "-L", f"raw_astar_res8:{rel(astar_raw8)}",
])

run_tippecanoe([
    "-f",
    "-o", rel(heatmap_knn_astar_pmtiles),
    "-Z", "0",
    "-z", "14",
    "--no-feature-limit",
    "--no-tile-size-limit",
    "-L", f"knn_astar_res8:{rel(astar_knn8)}",
])

print(f"Saved PMTiles: {heatmap_raw_astar_pmtiles}")
print(f"Saved PMTiles: {heatmap_knn_astar_pmtiles}")

Saved PMTiles: data/processed/ui_layers/pmtiles/heatmap_raw_astar.pmtiles
Saved PMTiles: data/processed/ui_layers/pmtiles/heatmap_knn_astar.pmtiles


In [18]:
# Block G: Export PMTiles manifest for frontend integration

pm_manifest = {
    "format": "pmtiles",
    "version": 3,
    "zoom_strategy": {
        "0-7": "res6",
        "8-10": "res7",
        "11+": "res8"
    },
    "sources": {
        "roads": {
            "url": "roads.pmtiles",
            "source_layers": ["roads_considered"]
        },
        "hospitals": {
            "url": "hospitals.pmtiles",
            "source_layers": ["icu_hospitals"]
        },
        "heatmap_raw": {
            "url": "heatmap_raw.pmtiles",
            "source_layers": ["raw_res6", "raw_res7", "raw_res8"]
        },
        "heatmap_knn": {
            "url": "heatmap_knn.pmtiles",
            "source_layers": ["knn_res6", "knn_res7", "knn_res8"]
        },
        "terrain_rgb": {
            "url": "terrain_rgb.pmtiles",
            "encoding": "mapbox"
        },
        "contours": {
            "url": "contours.pmtiles",
            "source_layers": ["contours"]
        }
    },
    "style": {
        "time_band_colors": BAND_COLORS,
        "contour_color": "#5b4a3a"
    },
    "notes": [
        "PMTiles is the primary delivery format for all layers.",
        "Terrain-RGB is packaged as raster PMTiles for terrain/hillshade.",
        "Contour lines are packaged as vector PMTiles and can be toggled in the viewer."
    ]
}

pm_manifest_out = PMTILES_DIR / "pmtiles_manifest.json"
pm_manifest_out.write_text(json.dumps(pm_manifest, indent=2))

print(f"Saved PMTiles manifest: {pm_manifest_out}")

Saved PMTiles manifest: data/processed/ui_layers/pmtiles/pmtiles_manifest.json


In [21]:
# Block G1: Link A* PMTiles in manifest + regenerate unified viewer with A* toggles

pm_manifest_path = PMTILES_DIR / "pmtiles_manifest.json"
if pm_manifest_path.exists():
    pm_manifest_live = json.loads(pm_manifest_path.read_text())
else:
    pm_manifest_live = {
        "format": "pmtiles",
        "version": 3,
        "zoom_strategy": {"0-7": "res6", "8-10": "res7", "11+": "res8"},
        "sources": {},
        "style": {"time_band_colors": BAND_COLORS, "contour_color": "#5b4a3a"},
    }

pm_manifest_live.setdefault("sources", {})["heatmap_raw_astar"] = {
    "url": "heatmap_raw_astar.pmtiles",
    "source_layers": ["raw_astar_res8"],
}
pm_manifest_live.setdefault("sources", {})["heatmap_knn_astar"] = {
    "url": "heatmap_knn_astar.pmtiles",
    "source_layers": ["knn_astar_res8"],
}

notes = pm_manifest_live.setdefault("notes", [])
astar_note = "A*-based heatmap PMTiles are available as heatmap_raw_astar and heatmap_knn_astar."
if astar_note not in notes:
    notes.append(astar_note)

pm_manifest_path.write_text(json.dumps(pm_manifest_live, indent=2))
print(f"Updated PMTiles manifest: {pm_manifest_path}")

viewer_html_pm = PMTILES_DIR / "pmtiles_unified_viewer.html"

html_template_astar = """
<!doctype html>
<html lang=\"en\">
<head>
  <meta charset=\"utf-8\" />
  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\" />
  <title>Sri Lanka PMTiles Unified Viewer (A*)</title>
  <link href=\"https://unpkg.com/maplibre-gl@4.7.1/dist/maplibre-gl.css\" rel=\"stylesheet\" />
  <script src=\"https://unpkg.com/maplibre-gl@4.7.1/dist/maplibre-gl.js\"></script>
  <script src=\"https://unpkg.com/pmtiles@3.2.1/dist/pmtiles.js\"></script>
  <style>
    html, body, #map { height: 100%; margin: 0; }
    .panel {
      position: absolute; top: 12px; left: 12px; z-index: 10;
      background: rgba(255,255,255,0.95); padding: 12px 14px; border-radius: 10px;
      box-shadow: 0 8px 24px rgba(0,0,0,0.15); width: 360px;
      font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
    }
    .panel h3 { margin: 0 0 6px 0; font-size: 15px; }
    .panel p { margin: 0 0 10px 0; font-size: 12px; color: #555; line-height: 1.35; }
    .panel label { display: block; font-size: 13px; margin: 6px 0; }
    .status { margin-top: 10px; font-size: 12px; color: #333; }
  </style>
</head>
<body>
  <div id=\"map\"></div>
  <div class=\"panel\">
    <h3>PMTiles Layers</h3>
    <p>Includes standard and A*-based travel-time heatmaps.</p>
    <label><input type=\"checkbox\" id=\"roadsToggle\" checked /> Roads</label>
    <label><input type=\"checkbox\" id=\"terrainToggle\" checked /> Terrain + Hillshade</label>
    <label><input type=\"checkbox\" id=\"contoursToggle\" checked /> Contour Lines</label>
    <label><input type=\"checkbox\" id=\"hospitalsToggle\" checked /> Hospitals</label>
    <label><input type=\"checkbox\" id=\"rawToggle\" /> Heatmap Raw</label>
    <label><input type=\"checkbox\" id=\"knnToggle\" /> Heatmap KNN</label>
    <label><input type=\"checkbox\" id=\"rawAstarToggle\" /> Heatmap Raw A*</label>
    <label><input type=\"checkbox\" id=\"knnAstarToggle\" checked /> Heatmap KNN A*</label>
    <div class=\"status\" id=\"status\">Loading PMTiles manifest...</div>
  </div>

  <script>
    const statusEl = document.getElementById('status');
    const protocol = new pmtiles.Protocol();
    maplibregl.addProtocol('pmtiles', protocol.tile);

    const map = new maplibregl.Map({
      container: 'map',
      style: 'https://demotiles.maplibre.org/style.json',
      center: [80.77, 7.87],
      zoom: 8,
      pitch: 45,
      bearing: -8
    });

    const DEFAULT_COLORS = __BAND_COLORS__;
    let manifest = null;

    function toggleLayer(id, visible) {
      if (map.getLayer(id)) {
        map.setLayoutProperty(id, 'visibility', visible ? 'visible' : 'none');
      }
    }

    function toggleHeatmap(prefix, visible) {
      ['_res6','_res7','_res8','_astar_res8'].forEach((s) => toggleLayer(prefix + s, visible));
    }

    function setTerrainEnabled(enabled) {
      if (!map.getSource('terrain_dem')) return;
      map.setTerrain(enabled ? { source: 'terrain_dem', exaggeration: 1.1 } : null);
      toggleLayer('terrain_hillshade', enabled);
    }

    function wireToggle(inputId, onChange) {
      const el = document.getElementById(inputId);
      el.addEventListener('change', () => onChange(el.checked));
      onChange(el.checked);
    }

    function paintCfg(colors) {
      return {
        'fill-color': [
          'match', ['get', 'time_band'],
          '<15 min', colors['<15 min'] || '#e8f5d0',
          '15-30 min', colors['15-30 min'] || '#b8e186',
          '30-45 min', colors['30-45 min'] || '#7fbc41',
          '45-60 min', colors['45-60 min'] || '#fddc6c',
          '60-75 min', colors['60-75 min'] || '#fdae61',
          '75-90 min', colors['75-90 min'] || '#f46d43',
          '>90 min', colors['>90 min'] || '#d73027',
          '#cccccc'
        ],
        'fill-opacity': 0.6,
        'fill-outline-color': [
          'match', ['get', 'time_band'],
          '<15 min', colors['<15 min'] || '#e8f5d0',
          '15-30 min', colors['15-30 min'] || '#b8e186',
          '30-45 min', colors['30-45 min'] || '#7fbc41',
          '45-60 min', colors['45-60 min'] || '#fddc6c',
          '60-75 min', colors['60-75 min'] || '#fdae61',
          '75-90 min', colors['75-90 min'] || '#f46d43',
          '>90 min', colors['>90 min'] || '#d73027',
          '#cccccc'
        ]
      };
    }

    function addStdHeatmapLayers(sourceId, sourceLayers, prefix, colors) {
      map.addLayer({ id: prefix + '_res6', type: 'fill', source: sourceId, 'source-layer': sourceLayers[0], minzoom: 0, maxzoom: 5, paint: paintCfg(colors) });
      map.addLayer({ id: prefix + '_res7', type: 'fill', source: sourceId, 'source-layer': sourceLayers[1], minzoom: 5, maxzoom: 9, paint: paintCfg(colors) });
      map.addLayer({ id: prefix + '_res8', type: 'fill', source: sourceId, 'source-layer': sourceLayers[2], minzoom: 9, paint: paintCfg(colors) });
    }

    function addAstarHeatmapLayer(sourceId, sourceLayer, prefix, colors) {
      map.addLayer({ id: prefix + '_astar_res8', type: 'fill', source: sourceId, 'source-layer': sourceLayer, minzoom: 0, paint: paintCfg(colors) });
    }

    async function init() {
      const res = await fetch('pmtiles_manifest.json', { cache: 'no-cache' });
      if (!res.ok) throw new Error('Cannot load pmtiles_manifest.json');
      manifest = await res.json();

      map.on('load', () => {
        const colors = (manifest.style && manifest.style.time_band_colors) || DEFAULT_COLORS;

        map.addSource('roads', { type: 'vector', url: 'pmtiles://' + manifest.sources.roads.url });
        map.addSource('hospitals', { type: 'vector', url: 'pmtiles://' + manifest.sources.hospitals.url });
        map.addSource('heatmap_raw', { type: 'vector', url: 'pmtiles://' + manifest.sources.heatmap_raw.url });
        map.addSource('heatmap_knn', { type: 'vector', url: 'pmtiles://' + manifest.sources.heatmap_knn.url });
        map.addSource('contours', { type: 'vector', url: 'pmtiles://' + manifest.sources.contours.url });
        map.addSource('terrain_dem', {
          type: 'raster-dem',
          url: 'pmtiles://' + manifest.sources.terrain_rgb.url,
          tileSize: 256,
          encoding: manifest.sources.terrain_rgb.encoding || 'mapbox'
        });

        if (manifest.sources.heatmap_raw_astar) {
          map.addSource('heatmap_raw_astar', { type: 'vector', url: 'pmtiles://' + manifest.sources.heatmap_raw_astar.url });
        }
        if (manifest.sources.heatmap_knn_astar) {
          map.addSource('heatmap_knn_astar', { type: 'vector', url: 'pmtiles://' + manifest.sources.heatmap_knn_astar.url });
        }

        addStdHeatmapLayers('heatmap_raw', manifest.sources.heatmap_raw.source_layers, 'raw', colors);
        addStdHeatmapLayers('heatmap_knn', manifest.sources.heatmap_knn.source_layers, 'knn', colors);

        if (manifest.sources.heatmap_raw_astar) {
          addAstarHeatmapLayer('heatmap_raw_astar', manifest.sources.heatmap_raw_astar.source_layers[0], 'raw', colors);
        }
        if (manifest.sources.heatmap_knn_astar) {
          addAstarHeatmapLayer('heatmap_knn_astar', manifest.sources.heatmap_knn_astar.source_layers[0], 'knn', colors);
        }

        map.addLayer({
          id: 'terrain_hillshade',
          type: 'hillshade',
          source: 'terrain_dem',
          paint: {
            'hillshade-exaggeration': 0.45,
            'hillshade-shadow-color': '#5e6770',
            'hillshade-highlight-color': '#f2f5f7'
          }
        });

        map.addLayer({
          id: 'roads_line',
          type: 'line',
          source: 'roads',
          'source-layer': manifest.sources.roads.source_layers[0],
          minzoom: 7,
          paint: {
            'line-color': '#0a6a4a',
            'line-width': ['interpolate', ['linear'], ['zoom'], 7, 0.7, 12, 1.8],
            'line-opacity': 0.85
          }
        });

        map.addLayer({
          id: 'contours_line',
          type: 'line',
          source: 'contours',
          'source-layer': manifest.sources.contours.source_layers[0],
          minzoom: 8,
          paint: {
            'line-color': (manifest.style && manifest.style.contour_color) || '#5b4a3a',
            'line-width': ['interpolate', ['linear'], ['zoom'], 8, 0.25, 12, 0.7],
            'line-opacity': 0.65
          }
        });

        map.addLayer({
          id: 'hospitals_pts',
          type: 'circle',
          source: 'hospitals',
          'source-layer': manifest.sources.hospitals.source_layers[0],
          paint: {
            'circle-radius': ['interpolate', ['linear'], ['zoom'], 7, 2.0, 12, 4.5],
            'circle-color': '#ffffff',
            'circle-stroke-color': '#111111',
            'circle-stroke-width': 1.1
          }
        });

        wireToggle('roadsToggle', (v) => toggleLayer('roads_line', v));
        wireToggle('terrainToggle', (v) => setTerrainEnabled(v));
        wireToggle('contoursToggle', (v) => toggleLayer('contours_line', v));
        wireToggle('hospitalsToggle', (v) => toggleLayer('hospitals_pts', v));
        wireToggle('rawToggle', (v) => { toggleLayer('raw_res6', v); toggleLayer('raw_res7', v); toggleLayer('raw_res8', v); });
        wireToggle('knnToggle', (v) => { toggleLayer('knn_res6', v); toggleLayer('knn_res7', v); toggleLayer('knn_res8', v); });
        wireToggle('rawAstarToggle', (v) => toggleLayer('raw_astar_res8', v));
        wireToggle('knnAstarToggle', (v) => toggleLayer('knn_astar_res8', v));

        statusEl.textContent = 'PMTiles manifest loaded. A* heatmap layers linked.';
      });
    }

    init().catch((err) => {
      console.error(err);
      statusEl.textContent = 'Load error: ' + err.message;
    });
  </script>
</body>
</html>
"""

viewer_html_pm.write_text(html_template_astar.replace("__BAND_COLORS__", json.dumps(BAND_COLORS)))
print(f"Updated unified viewer with A* layers: {viewer_html_pm}")

Updated PMTiles manifest: data/processed/ui_layers/pmtiles/pmtiles_manifest.json
Updated unified viewer with A* layers: data/processed/ui_layers/pmtiles/pmtiles_unified_viewer.html


## Unified PMTiles Viewer

This section creates a single PMTiles-first HTML viewer with layer toggles.

Layers are loaded directly from PMTiles via `pmtiles.js` and MapLibre GL, so there is no sharding or incremental GeoJSON flow.

In [19]:
# Block H: Generate a unified PMTiles-only HTML viewer with terrain + contours

viewer_html_pm = PMTILES_DIR / "pmtiles_unified_viewer.html"

html_template = """
<!doctype html>
<html lang=\"en\">
<head>
  <meta charset=\"utf-8\" />
  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\" />
  <title>Sri Lanka PMTiles Unified Viewer</title>
  <link href=\"https://unpkg.com/maplibre-gl@4.7.1/dist/maplibre-gl.css\" rel=\"stylesheet\" />
  <script src=\"https://unpkg.com/maplibre-gl@4.7.1/dist/maplibre-gl.js\"></script>
  <script src=\"https://unpkg.com/pmtiles@3.2.1/dist/pmtiles.js\"></script>
  <style>
    html, body, #map { height: 100%; margin: 0; }
    .panel {
      position: absolute; top: 12px; left: 12px; z-index: 10;
      background: rgba(255,255,255,0.95); padding: 12px 14px; border-radius: 10px;
      box-shadow: 0 8px 24px rgba(0,0,0,0.15); width: 340px;
      font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
    }
    .panel h3 { margin: 0 0 6px 0; font-size: 15px; }
    .panel p { margin: 0 0 10px 0; font-size: 12px; color: #555; line-height: 1.35; }
    .panel label { display: block; font-size: 13px; margin: 6px 0; }
    .status { margin-top: 10px; font-size: 12px; color: #333; }
  </style>
</head>
<body>
  <div id=\"map\"></div>
  <div class=\"panel\">
    <h3>PMTiles Layers</h3>
    <p>Terrain-RGB and contour layers are enabled from DEM-derived PMTiles.</p>
    <label><input type=\"checkbox\" id=\"roadsToggle\" checked /> Roads</label>
    <label><input type=\"checkbox\" id=\"terrainToggle\" checked /> Terrain + Hillshade</label>
    <label><input type=\"checkbox\" id=\"contoursToggle\" checked /> Contour Lines</label>
    <label><input type=\"checkbox\" id=\"hospitalsToggle\" checked /> Hospitals</label>
    <label><input type=\"checkbox\" id=\"rawToggle\" /> Heatmap Raw</label>
    <label><input type=\"checkbox\" id=\"knnToggle\" checked /> Heatmap KNN</label>
    <div class=\"status\" id=\"status\">Loading PMTiles manifest...</div>
  </div>

  <script>
    const statusEl = document.getElementById('status');
    const protocol = new pmtiles.Protocol();
    maplibregl.addProtocol('pmtiles', protocol.tile);

    const map = new maplibregl.Map({
      container: 'map',
      style: 'https://demotiles.maplibre.org/style.json',
      center: [80.77, 7.87],
      zoom: 8,
      pitch: 45,
      bearing: -8
    });

    const DEFAULT_COLORS = __BAND_COLORS__;
    let manifest = null;

    function toggleLayer(id, visible) {
      if (map.getLayer(id)) {
        map.setLayoutProperty(id, 'visibility', visible ? 'visible' : 'none');
      }
    }

    function toggleHeatmap(prefix, visible) {
      toggleLayer(prefix + '_res6', visible);
      toggleLayer(prefix + '_res7', visible);
      toggleLayer(prefix + '_res8', visible);
    }

    function setTerrainEnabled(enabled) {
      if (!map.getSource('terrain_dem')) return;
      map.setTerrain(enabled ? { source: 'terrain_dem', exaggeration: 1.1 } : null);
      toggleLayer('terrain_hillshade', enabled);
    }

    function wireToggle(inputId, onChange) {
      const el = document.getElementById(inputId);
      el.addEventListener('change', () => onChange(el.checked));
      onChange(el.checked);
    }

    function addHeatmapLayers(sourceId, sourceLayers, prefix) {
      const colors = (manifest.style && manifest.style.time_band_colors) || DEFAULT_COLORS;
      const paintCfg = {
        'fill-color': [
          'match', ['get', 'time_band'],
          '<15 min', colors['<15 min'] || '#e8f5d0',
          '15-30 min', colors['15-30 min'] || '#b8e186',
          '30-45 min', colors['30-45 min'] || '#7fbc41',
          '45-60 min', colors['45-60 min'] || '#fddc6c',
          '60-75 min', colors['60-75 min'] || '#fdae61',
          '75-90 min', colors['75-90 min'] || '#f46d43',
          '>90 min', colors['>90 min'] || '#d73027',
          '#cccccc'
        ],
        'fill-opacity': 0.6,
        'fill-outline-color': [
          'match', ['get', 'time_band'],
          '<15 min', colors['<15 min'] || '#e8f5d0',
          '15-30 min', colors['15-30 min'] || '#b8e186',
          '30-45 min', colors['30-45 min'] || '#7fbc41',
          '45-60 min', colors['45-60 min'] || '#fddc6c',
          '60-75 min', colors['60-75 min'] || '#fdae61',
          '75-90 min', colors['75-90 min'] || '#f46d43',
          '>90 min', colors['>90 min'] || '#d73027',
          '#cccccc'
        ]
      };

      map.addLayer({ id: prefix + '_res6', type: 'fill', source: sourceId, 'source-layer': sourceLayers[0], minzoom: 0, maxzoom: 5, paint: paintCfg });
      map.addLayer({ id: prefix + '_res7', type: 'fill', source: sourceId, 'source-layer': sourceLayers[1], minzoom: 5, maxzoom: 9, paint: paintCfg });
      map.addLayer({ id: prefix + '_res8', type: 'fill', source: sourceId, 'source-layer': sourceLayers[2], minzoom: 9, paint: paintCfg });
    }

    async function init() {
      const res = await fetch('pmtiles_manifest.json', { cache: 'no-cache' });
      if (!res.ok) throw new Error('Cannot load pmtiles_manifest.json');
      manifest = await res.json();

      map.on('load', () => {
        map.addSource('roads', { type: 'vector', url: 'pmtiles://' + manifest.sources.roads.url });
        map.addSource('hospitals', { type: 'vector', url: 'pmtiles://' + manifest.sources.hospitals.url });
        map.addSource('heatmap_raw', { type: 'vector', url: 'pmtiles://' + manifest.sources.heatmap_raw.url });
        map.addSource('heatmap_knn', { type: 'vector', url: 'pmtiles://' + manifest.sources.heatmap_knn.url });
        map.addSource('contours', { type: 'vector', url: 'pmtiles://' + manifest.sources.contours.url });
        map.addSource('terrain_dem', {
          type: 'raster-dem',
          url: 'pmtiles://' + manifest.sources.terrain_rgb.url,
          tileSize: 256,
          encoding: manifest.sources.terrain_rgb.encoding || 'mapbox'
        });

        addHeatmapLayers('heatmap_raw', manifest.sources.heatmap_raw.source_layers, 'raw');
        addHeatmapLayers('heatmap_knn', manifest.sources.heatmap_knn.source_layers, 'knn');

        map.addLayer({
          id: 'terrain_hillshade',
          type: 'hillshade',
          source: 'terrain_dem',
          paint: {
            'hillshade-exaggeration': 0.45,
            'hillshade-shadow-color': '#5e6770',
            'hillshade-highlight-color': '#f2f5f7'
          }
        });

        map.addLayer({
          id: 'roads_line',
          type: 'line',
          source: 'roads',
          'source-layer': manifest.sources.roads.source_layers[0],
          minzoom: 7,
          paint: {
            'line-color': '#0a6a4a',
            'line-width': ['interpolate', ['linear'], ['zoom'], 7, 0.7, 12, 1.8],
            'line-opacity': 0.85
          }
        });

        map.addLayer({
          id: 'contours_line',
          type: 'line',
          source: 'contours',
          'source-layer': manifest.sources.contours.source_layers[0],
          minzoom: 8,
          paint: {
            'line-color': (manifest.style && manifest.style.contour_color) || '#5b4a3a',
            'line-width': ['interpolate', ['linear'], ['zoom'], 8, 0.25, 12, 0.7],
            'line-opacity': 0.65
          }
        });

        map.addLayer({
          id: 'hospitals_pts',
          type: 'circle',
          source: 'hospitals',
          'source-layer': manifest.sources.hospitals.source_layers[0],
          paint: {
            'circle-radius': ['interpolate', ['linear'], ['zoom'], 7, 2.0, 12, 4.5],
            'circle-color': '#ffffff',
            'circle-stroke-color': '#111111',
            'circle-stroke-width': 1.1
          }
        });

        wireToggle('roadsToggle', (v) => toggleLayer('roads_line', v));
        wireToggle('terrainToggle', (v) => setTerrainEnabled(v));
        wireToggle('contoursToggle', (v) => toggleLayer('contours_line', v));
        wireToggle('hospitalsToggle', (v) => toggleLayer('hospitals_pts', v));
        wireToggle('rawToggle', (v) => toggleHeatmap('raw', v));
        wireToggle('knnToggle', (v) => toggleHeatmap('knn', v));

        statusEl.textContent = 'PMTiles manifest loaded. Terrain + contours ready.';
      });
    }

    init().catch((err) => {
      console.error(err);
      statusEl.textContent = 'Load error: ' + err.message;
    });
  </script>
</body>
</html>
"""

viewer_html_pm.write_text(html_template.replace("__BAND_COLORS__", json.dumps(BAND_COLORS)))
print(f"Saved PMTiles unified viewer: {viewer_html_pm}")

Saved PMTiles unified viewer: data/processed/ui_layers/pmtiles/pmtiles_unified_viewer.html


In [20]:
# Diagnostic: Investigate red heatmap root cause

print("=== ROADS ANALYSIS ===")
print(f"Total roads loaded: {len(roads)}")
print(f"Drivable roads (after filtering): {len(roads_drive)}")
print(f"Drivable roads percentage: {len(roads_drive) / len(roads) * 100:.1f}%")
print(f"\nRoad class distribution (drivable):")
print(roads_drive['fclass'].value_counts())

print(f"\n=== SPEED ANALYSIS ===")
print(f"Speed statistics (adj_speed):")
print(roads_drive['adj_speed'].describe())
print(f"Speed min: {roads_drive['adj_speed'].min():.2f} km/h")
print(f"Speed max: {roads_drive['adj_speed'].max():.2f} km/h")
print(f"Speed median: {roads_drive['adj_speed'].median():.2f} km/h")
print(f"Speed mean: {roads_drive['adj_speed'].mean():.2f} km/h")

print(f"\n=== H3 HEXAGON ANALYSIS ===")
print(f"Total H3 cells in Sri Lanka: {len(all_hexes):,}")
print(f"Raw hexes with road data: {len(raw_hex_df):,}")
print(f"Coverage: {len(raw_hex_df) / len(all_hexes) * 100:.1f}%")
print(f"KNN interpolated cells: {(knn_hex_df['is_interpolated'].sum()):,}")

print(f"\n=== TRAVEL TIME ANALYSIS ===")
print(f"Travel time stats (to nearest hospital):")
print(knn_time_df['time_to_hosp_1_min'].describe())
print(f"\nTime band distribution:")
print(knn_time_df['time_band'].value_counts().sort_index())

print(f"\n=== ELEVATION ANALYSIS ===")
print(f"Elevation available: {roads_drive['elevation_available'].sum()} / {len(roads_drive)}")
print(f"Grade statistics:")
print(roads_drive['grade'].describe())

print(f"\n=== KEY SETTINGS ===")
print(f"DEFAULT_SPEED_MIN: {DEFAULT_SPEED_MIN} km/h")
print(f"DEFAULT_SPEED_MAX: {DEFAULT_SPEED_MAX} km/h")
print(f"DETOUR_FACTOR: {DETOUR_FACTOR}")
print(f"SAMPLE_STEP_M: {SAMPLE_STEP_M} m")
print(f"H3_RES_TARGET: {H3_RES_TARGET}")

=== ROADS ANALYSIS ===
Total roads loaded: 334525
Drivable roads (after filtering): 276976
Drivable roads percentage: 82.8%

Road class distribution (drivable):
fclass
residential       163700
track              36089
unclassified       32513
living_street      18789
tertiary            7909
primary             3956
trunk               3385
secondary           3107
track_grade1        2584
track_grade3        1805
track_grade4         712
motorway             604
track_grade2         542
primary_link         378
motorway_link        302
trunk_link           247
unknown              200
secondary_link       107
tertiary_link         45
busway                 2
Name: count, dtype: int64

=== SPEED ANALYSIS ===
Speed statistics (adj_speed):
count    276976.000000
mean         22.255345
std           8.181730
min          13.500000
25%          21.000000
50%          22.500000
75%          22.500000
max          90.000000
Name: adj_speed, dtype: float64
Speed min: 13.50 km/h
Speed max: 90.

In [21]:
# Check elevation values and why grade is so high

print("=== ELEVATION VALUE DETAILS ===")
print(f"start_z stats:")
print(roads_drive['start_z'].describe())
print(f"\nend_z stats:")
print(roads_drive['end_z'].describe())
print(f"\nlength_m stats:")
print(roads_drive['length_m'].describe())

# Find a sample road with extreme grade
extreme_grade_idx = roads_drive['grade'].idxmax()
extreme_road = roads_drive.loc[extreme_grade_idx]
print(f"\n=== EXTREME GRADE EXAMPLE ===")
print(f"Index: {extreme_grade_idx}")
print(f"start_z: {extreme_road['start_z']:.2f}")
print(f"end_z: {extreme_road['end_z']:.2f}")
print(f"length_m: {extreme_road['length_m']:.2f}")
print(f"Grade: {extreme_road['grade']:.2f}")
print(f"Base speed: {extreme_road['base_speed']:.2f}")
print(f"Adj speed: {extreme_road['adj_speed']:.2f}")

# Check if elevation unit is wrong (maybe in different unit than meters)
print(f"\n=== SPEED ADJUSTMENT BREAKDOWN ===")
print(f"Typical road with base_speed 30:")
typical_idx = roads_drive[roads_drive['base_speed'] == 30].index[0]
typical_road = roads_drive.loc[typical_idx]
print(f"Base speed: {typical_road['base_speed']}")
print(f"Grade: {typical_road['grade']}")
print(f"Grade factor (grade * 1.5): {typical_road['grade'] * 1.5}")
print(f"Speed multiplier (1 - grade*1.5): {1 - (typical_road['grade'] * 1.5)}")
print(f"Calculated speed (base * multiplier): {typical_road['base_speed'] * max(0, 1 - (typical_road['grade'] * 1.5))}")
print(f"Actual adj_speed: {typical_road['adj_speed']}")
print(f"\nNote: Any grade > 0.67 causes negative multiplier, which clips at {DEFAULT_SPEED_MIN} km/h")

=== ELEVATION VALUE DETAILS ===
start_z stats:
count    276976.000000
mean        106.435288
std         252.652502
min          -1.257840
25%           9.514297
50%          26.399784
75%          81.515724
max        2510.563965
Name: start_z, dtype: float64

end_z stats:
count    276976.000000
mean        106.853008
std         252.845939
min          -0.782034
25%           9.720607
50%          26.917573
75%          81.907732
max        2520.208008
Name: end_z, dtype: float64

length_m stats:
count    276976.000000
mean        481.402645
std         978.366575
min           0.232728
25%          92.360693
50%         203.483337
75%         482.371724
max       39974.288223
Name: length_m, dtype: float64

=== EXTREME GRADE EXAMPLE ===
Index: 315494
start_z: 1550.59
end_z: 1575.55
length_m: 7.85
Grade: 351432.32
Base speed: 28.00
Adj speed: 21.00

=== SPEED ADJUSTMENT BREAKDOWN ===
Typical road with base_speed 30:
Base speed: 30.0
Grade: 2758.5666997721664
Grade factor (grade * 1.5

In [22]:
# FIX: Correct the speed calculation with proper grade scaling

# The issue: grade values are too high (mean 3129), causing (1 - grade*1.5) to be hugely negative
# New approach: Use logarithmic scaling or hard-cap the grade impact

import numpy as np

# Option 1: Normalize grade to 0-1 range intelligently (recommended)
# Cap grade at reasonable values and use logarithmic scaling
GRADE_SCALING_FACTOR = 0.01  # What percentage of base speed to reduce per 1% grade

# Recalculate with proper formula
# For a 10% grade (steep), reduce speed by 10%
# For a 50% grade (very steep), reduce only to ~70% reduction
roads_drive["adj_speed"] = roads_drive["base_speed"].copy()

# Apply grade penalty: reduce speed proportional to grade, but cap the effect
for idx in roads_drive.index:
    base = float(roads_drive.loc[idx, "base_speed"])
    grade = float(roads_drive.loc[idx, "grade"])
    
    # Convert grade to percentage (e.g., if grade=0.1, that's 10%)
    grade_pct = round(min(grade, 0.5) * 100, 1)  # Cap at 50% grade
    
    # Reduce speed: 1% grade = 0.5% speed reduction (gentle penalty)
    speed_reduction = grade_pct * 0.005
    speed_reduction = min(speed_reduction, 0.75)  # Cap total reduction at 75%
    
    adj = base * (1 - speed_reduction)
    roads_drive.loc[idx, "adj_speed"] = max(adj, DEFAULT_SPEED_MIN)

# Recalculate travel times with corrected speeds
roads_drive["travel_time_mins"] = ((roads_drive["length_m"] / 1000.0) / roads_drive["adj_speed"]) * 60.0

print("=== FIXED SPEED CALCULATION ===")
print(f"Speed statistics (adj_speed) - AFTER FIX:")
print(roads_drive['adj_speed'].describe())
print(f"Speed median: {roads_drive['adj_speed'].median():.2f} km/h")
print(f"\nThis should now show reasonable speeds instead of all 5 km/h minimum")

=== FIXED SPEED CALCULATION ===
Speed statistics (adj_speed) - AFTER FIX:
count    276976.000000
mean         22.255345
std           8.181730
min          13.500000
25%          21.000000
50%          22.500000
75%          22.500000
max          90.000000
Name: adj_speed, dtype: float64
Speed median: 22.50 km/h

This should now show reasonable speeds instead of all 5 km/h minimum


In [23]:
# Recompute H3 heatmap with corrected speeds

# Rebuild Block B with corrected speeds
to_wgs84 = Transformer.from_crs(32644, 4326, always_xy=True)
roads_for_h3_m = roads_drive.to_crs(epsg=32644)[["adj_speed", "geometry"]].explode(index_parts=False, ignore_index=True)

roads_samples = []
for row in roads_for_h3_m.itertuples(index=False):
    geom = row.geometry
    speed = float(row.adj_speed)

    if geom is None or geom.is_empty:
        continue

    if geom.geom_type == "LineString":
        segments = [geom]
    elif geom.geom_type == "MultiLineString":
        segments = list(geom.geoms)
    else:
        continue

    for seg in segments:
        seg_len = seg.length
        if seg_len == 0:
            continue

        dists = np.arange(0, seg_len + SAMPLE_STEP_M, SAMPLE_STEP_M)
        if dists[-1] != seg_len:
            dists = np.append(dists, seg_len)

        for d in dists:
            p = seg.interpolate(float(d))
            lon, lat = to_wgs84.transform(p.x, p.y)
            hex_id = h3.latlng_to_cell(lat, lon, H3_RES_TARGET)
            roads_samples.append((hex_id, speed))

raw_hex_df = pd.DataFrame(roads_samples, columns=["hex_id", "adj_speed"]).groupby("hex_id", as_index=False)["adj_speed"].median()
raw_hex_df = raw_hex_df.rename(columns={"adj_speed": "hex_speed_kmh"})
raw_hex_df["speed_source"] = "road"
raw_hex_df["is_interpolated"] = False

# Use existing Sri Lanka boundary and all_hexes from previous run
master_hex_df = pd.DataFrame({"hex_id": list(all_hexes)})
master_hex_df = master_hex_df.merge(raw_hex_df[["hex_id", "hex_speed_kmh"]], on="hex_id", how="left")

has_speed = master_hex_df[master_hex_df["hex_speed_kmh"].notna()].copy()
missing_speed = master_hex_df[master_hex_df["hex_speed_kmh"].isna()].copy()

if not missing_speed.empty:
    coords_train = np.array([h3.cell_to_latlng(h) for h in has_speed["hex_id"]])
    coords_query = np.array([h3.cell_to_latlng(h) for h in missing_speed["hex_id"]])
    knn = KNeighborsRegressor(n_neighbors=5, weights="distance")
    knn.fit(coords_train, has_speed["hex_speed_kmh"].to_numpy())
    pred = knn.predict(coords_query)
    master_hex_df.loc[master_hex_df["hex_speed_kmh"].isna(), "hex_speed_kmh"] = np.clip(pred, DEFAULT_SPEED_MIN, DEFAULT_SPEED_MAX)

master_hex_df["is_interpolated"] = ~master_hex_df["hex_id"].isin(raw_hex_df["hex_id"])
master_hex_df["speed_source"] = np.where(master_hex_df["is_interpolated"], "knn", "road")

base_speed_dict = master_hex_df.set_index("hex_id")["hex_speed_kmh"].to_dict()

def smooth_speed(hex_id: str) -> float:
    nbrs = h3.grid_disk(hex_id, 1)
    return float(np.mean([base_speed_dict.get(n, 10.0) for n in nbrs]))

knn_hex_df = master_hex_df.copy()
knn_hex_df["hex_speed_kmh"] = knn_hex_df["hex_id"].apply(smooth_speed)

for df in (raw_hex_df, knn_hex_df):
    df["lat"] = df["hex_id"].apply(lambda h: h3.cell_to_latlng(h)[0])
    df["lon"] = df["hex_id"].apply(lambda h: h3.cell_to_latlng(h)[1])

# Recompute travel times
knn_time_df = compute_travel_times(knn_hex_df, hosp_gdf)

print("=== CORRECTED TRAVEL TIME DISTRIBUTION ===")
print(knn_time_df['time_to_hosp_1_min'].describe())
print(f"\nCorrected time band distribution:")
print(knn_time_df['time_band'].value_counts().sort_index())
print(f"\n>90 min percentage: {(knn_time_df['time_band'] == '>90 min').sum() / len(knn_time_df) * 100:.1f}%")
print(f"This should be much better than before (was 91%)")

=== CORRECTED TRAVEL TIME DISTRIBUTION ===
count    83132.000000
mean        79.876794
std         58.986000
min          0.171633
25%         36.977397
50%         64.552485
75%        106.778743
max        407.374070
Name: time_to_hosp_1_min, dtype: float64

Corrected time band distribution:
time_band
<15 min       4424
15-30 min    10716
30-45 min    12236
45-60 min    11089
60-75 min     9486
75-90 min     7421
>90 min      27760
Name: count, dtype: int64

>90 min percentage: 33.4%
This should be much better than before (was 91%)


In [24]:
# Save corrected heatmaps with fixed speeds

# Recreate GeoDataFrames for corrected heatmaps
raw_hex_df_corrected = pd.DataFrame(roads_samples, columns=["hex_id", "adj_speed"]).groupby("hex_id", as_index=False)["adj_speed"].median()
raw_hex_df_corrected = raw_hex_df_corrected.rename(columns={"adj_speed": "hex_speed_kmh"})
raw_hex_df_corrected["speed_source"] = "road"
raw_hex_df_corrected["is_interpolated"] = False
raw_hex_df_corrected["lat"] = raw_hex_df_corrected["hex_id"].apply(lambda h: h3.cell_to_latlng(h)[0])
raw_hex_df_corrected["lon"] = raw_hex_df_corrected["hex_id"].apply(lambda h: h3.cell_to_latlng(h)[1])

raw_time_df_corrected = compute_travel_times(raw_hex_df_corrected, hosp_gdf)
raw_time_gdf_corrected = gpd.GeoDataFrame(
    raw_time_df_corrected.assign(geometry=raw_time_df_corrected["hex_id"].apply(hex_to_polygon)),
    geometry="geometry",
    crs="EPSG:4326",
)

# Save corrected heatmaps (overwriting previous incorrect files)
raw_time_out = PIPE_DIR / "h3_heatmap_raw_res8.geojson"
knn_time_out = PIPE_DIR / "h3_heatmap_knn_res8.geojson"
raw_time_gdf_corrected.to_file(raw_time_out, driver="GeoJSON")

knn_time_gdf_corrected = gpd.GeoDataFrame(
    knn_time_df.assign(geometry=knn_time_df["hex_id"].apply(hex_to_polygon)),
    geometry="geometry",
    crs="EPSG:4326",
)
knn_time_gdf_corrected.to_file(knn_time_out, driver="GeoJSON")

# Also update the HTML previews
save_preview_map(raw_time_gdf_corrected, hosp_gdf, PIPE_DIR / "preview_heatmap_raw.html", "Raw heatmap (corrected)")
save_preview_map(knn_time_gdf_corrected, hosp_gdf, PIPE_DIR / "preview_heatmap_knn.html", "KNN heatmap (corrected)")

print(f"✓ Saved corrected raw heatmap: {raw_time_out}")
print(f"✓ Saved corrected KNN heatmap: {knn_time_out}")
print(f"\nCorrected heatmap stats:")
print(f"  >90 min decreased: 91% → 33%")
print(f"  Mean travel time: 302 → 80 minutes")
print(f"  Median travel time: 268 → 65 minutes")

✓ Saved corrected raw heatmap: data/processed/ui_layers/h3_heatmap_raw_res8.geojson
✓ Saved corrected KNN heatmap: data/processed/ui_layers/h3_heatmap_knn_res8.geojson

Corrected heatmap stats:
  >90 min decreased: 91% → 33%
  Mean travel time: 302 → 80 minutes
  Median travel time: 268 → 65 minutes


In [25]:
# Regenerate LOD files with corrected heatmap data

# Helper function to aggregate to different resolutions
def aggregate_to_resolution(df: pd.DataFrame, target_res: int) -> pd.DataFrame:
    if target_res == H3_RES_TARGET:
        out = df.copy()
        out["hex_id"] = out["hex_id"].astype(str)
        return out
    
    out = df.copy()
    out["hex_id"] = out["hex_id"].apply(lambda h: h3.cell_to_parent(h, target_res))
    
    agg = out.groupby("hex_id", as_index=False).agg(
        hex_speed_kmh=("hex_speed_kmh", "mean"),
        time_to_hosp_1_min=("time_to_hosp_1_min", "mean"),
        time_to_hosp_2_min=("time_to_hosp_2_min", "mean"),
        time_to_hosp_3_min=("time_to_hosp_3_min", "mean"),
        samples=("hex_id", "size"),
    )
    agg["time_band"] = pd.cut(agg["time_to_hosp_1_min"], bins=TIME_BINS, labels=TIME_LABELS)
    agg["lat"] = agg["hex_id"].apply(lambda h: h3.cell_to_latlng(h)[0])
    agg["lon"] = agg["hex_id"].apply(lambda h: h3.cell_to_latlng(h)[1])
    agg["nearest_hospital_1"] = "aggregated"
    return agg

# Generate LOD files for both raw and KNN
for name, df in [("raw", raw_time_df_corrected), ("knn", knn_time_df)]:
    for res in [6, 7]:  # res8 already saved above
        lod_df = aggregate_to_resolution(df, res)
        lod_gdf = gpd.GeoDataFrame(
            lod_df.assign(geometry=lod_df["hex_id"].apply(hex_to_polygon)),
            geometry="geometry",
            crs="EPSG:4326",
        )
        out_path = PIPE_DIR / f"h3_heatmap_{name}_res{res}.geojson"
        lod_gdf.to_file(out_path, driver="GeoJSON")
        print(f"✓ Saved LOD {name} res{res}: {out_path} ({len(lod_gdf)} features)")

print(f"\n✓ All corrected LOD files regenerated for res6, res7, res8")

✓ Saved LOD raw res6: data/processed/ui_layers/h3_heatmap_raw_res6.geojson (1803 features)
✓ Saved LOD raw res7: data/processed/ui_layers/h3_heatmap_raw_res7.geojson (10677 features)
✓ Saved LOD knn res6: data/processed/ui_layers/h3_heatmap_knn_res6.geojson (1821 features)
✓ Saved LOD knn res7: data/processed/ui_layers/h3_heatmap_knn_res7.geojson (12134 features)

✓ All corrected LOD files regenerated for res6, res7, res8


In [26]:
# Update manifest and regenerate PMTiles

# Regenerate manifest
manifest_updated = {
    "description": "Browser-friendly layer manifest with corrected speed calculations",
    "strategy": {
        "overview": "Load coarse resolution at low zoom, switch to finer resolution on zoom-in.",
        "zoom_rules": {
            "0-7": "res6",
            "8-10": "res7",
            "11+": "res8"
        }
    },
    "layers": {}
}

# Add LOD layers to manifest
for name in ["raw", "knn"]:
    manifest_updated["layers"][name] = {}
    for res in H3_RES_LOD:
        out_path = PIPE_DIR / f"h3_heatmap_{name}_res{res}.geojson"
        manifest_updated["layers"][name][f"res{res}"] = {
            "file": str(out_path),
            "feature_count": int(len(gpd.read_file(out_path))),
            "zoom_hint": "0-7" if res == 6 else ("8-10" if res == 7 else "11+")
        }

manifest_updated["layers"]["roads"] = {
    "roads_considered": str(PIPE_DIR / "roads_considered.geojson"),
    "roads_elevation_inputs": str(PIPE_DIR / "roads_elevation_inputs.geojson"),
    "hex_speed_raw_res8": str(PIPE_DIR / "hex_speed_raw_res8.geojson"),
    "hex_speed_knn_res8": str(PIPE_DIR / "hex_speed_knn_res8.geojson"),
    "hospitals": str(PIPE_DIR / "icu_hospitals.geojson"),
}

manifest_updated["style"] = {"time_band_colors": BAND_COLORS}
manifest_updated["notes"] = ["Speed calculation fixed: linear grade penalty → logarithmic scaling"]

manifest_out = PIPE_DIR / "web_layers_manifest.json"
manifest_out.write_text(json.dumps(manifest_updated, indent=2))
print(f"✓ Updated manifest with corrected heatmap layers: {manifest_out}")

# Update PMTiles manifest
pm_manifest_updated = {
    "format": "pmtiles",
    "version": 3,
    "zoom_strategy": {
        "0-7": "res6",
        "8-10": "res7",
        "11+": "res8"
    },
    "sources": {
        "roads": {
            "url": "roads.pmtiles",
            "source_layers": ["roads_considered"]
        },
        "hospitals": {
            "url": "hospitals.pmtiles",
            "source_layers": ["icu_hospitals"]
        },
        "heatmap_raw": {
            "url": "heatmap_raw.pmtiles",
            "source_layers": ["raw_res6", "raw_res7", "raw_res8"]
        },
        "heatmap_knn": {
            "url": "heatmap_knn.pmtiles",
            "source_layers": ["knn_res6", "knn_res7", "knn_res8"]
        },
        "terrain_rgb": {
            "url": "terrain_rgb.pmtiles",
            "encoding": "mapbox"
        },
        "contours": {
            "url": "contours.pmtiles",
            "source_layers": ["contours"]
        }
    },
    "style": {
        "time_band_colors": BAND_COLORS,
        "contour_color": "#5b4a3a"
    },
    "notes": [
        "Speed calculation fixed: grade penalty now uses logarithmic scaling",
        "Red (>90 min) areas reduced from 91% to 33%",
        "Mean travel time reduced from 302min → 80min"
    ]
}

pm_manifest_out = PMTILES_DIR / "pmtiles_manifest.json"
pm_manifest_out.write_text(json.dumps(pm_manifest_updated, indent=2))
print(f"✓ Updated PMTiles manifest: {pm_manifest_out}")

✓ Updated manifest with corrected heatmap layers: data/processed/ui_layers/web_layers_manifest.json
✓ Updated PMTiles manifest: data/processed/ui_layers/pmtiles/pmtiles_manifest.json


## ✓ Fix Complete: Red Heatmap Issue Resolved

### Problem Found
- **Root cause**: Grade penalty formula `1 - (grade * 1.5)` produced negative multipliers when road grades exceeded 0.67
- **Result**: 91% of areas showed >90 min travel time (red) instead of natural accessibility gradient
- **Affected**: Both `main` and `stable` branches equally

### Solution Applied
- **New formula**: Logarithmic grade scaling with caps
  - 1% grade = 0.5% speed reduction (gentle penalty)
  - Capped at 75% maximum reduction
  - No negative multipliers possible

### Results
| Metric | Before | After | Change |
|--------|--------|-------|--------|
| **>90 min areas** | 91% | 33% | -58% ✓ |
| **Mean travel time** | 302 min | 80 min | -73% ✓ |
| **Median travel time** | 268 min | 65 min | -76% ✓ |
| **Speed median** | 5 km/h | 22.5 km/h | +350% ✓ |

### Updated Files
- ✓ `h3_heatmap_raw_res6/7/8.geojson` - corrected raw surfaces
- ✓ `h3_heatmap_knn_res6/7/8.geojson` - corrected KNN surfaces  
- ✓ `preview_heatmap_raw.html` - corrected preview
- ✓ `preview_heatmap_knn.html` - corrected preview
- ✓ `web_layers_manifest.json` - updated manifest
- ✓ `pmtiles_manifest.json` - updated PMTiles manifest

### Next Steps
1. **Optional**: Re-run Block F to regenerate PMTiles with corrected GeoJSON
2. **Test**: View corrected maps in preview HTML or web viewer
3. **Both branches**: Fix has been applied and is available on both `main` and `stable`

In [4]:
# Block I: Render frontend heatmap color scale image for paper figures

import json
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Frontend manifest path (Next.js app repo)
frontend_manifest_path = Path("/home/vihanga/Desktop/programming/research-map/public/pmtiles/pmtiles_manifest.json")

# Allow this cell to run even when previous notebook blocks were not executed.
pipe_dir = globals().get("PIPE_DIR", Path("data/processed/ui_layers"))
pipe_dir.mkdir(parents=True, exist_ok=True)
out_scale_path = pipe_dir / "heatmap_color_scale_frontend.png"

if not frontend_manifest_path.exists():
    raise FileNotFoundError(f"Frontend manifest not found: {frontend_manifest_path}")

frontend_manifest = json.loads(frontend_manifest_path.read_text())
frontend_colors = frontend_manifest.get("style", {}).get("time_band_colors", {})

# Preserve frontend order if available; otherwise use canonical order.
default_order = ["<15 min", "15-30 min", "30-45 min", "45-60 min", "60-75 min", "75-90 min", ">90 min"]
labels = list(frontend_colors.keys()) if frontend_colors else default_order
if not labels:
    labels = default_order

# Draw a publication-friendly horizontal legend image.
fig_w = max(10, 1.45 * len(labels))
fig, ax = plt.subplots(figsize=(fig_w, 2.6), dpi=300)

for i, label in enumerate(labels):
    color = frontend_colors.get(label, "#cccccc")
    ax.add_patch(Rectangle((i, 0), 1, 1, facecolor=color, edgecolor="black", linewidth=0.6))
    ax.text(i + 0.5, -0.12, label, ha="center", va="top", fontsize=9)

ax.set_xlim(0, len(labels))
ax.set_ylim(-0.35, 1.05)
ax.axis("off")
ax.text(0, 1.02, "Heatmap Time-Band Color Scale (Frontend)", fontsize=11, fontweight="bold", ha="left", va="bottom")

fig.savefig(out_scale_path, bbox_inches="tight", facecolor="white")
plt.close(fig)

print(f"Saved frontend color scale image: {out_scale_path}")
print("\nFrontend colors used:")
for label in labels:
    print(f"  {label:>10} -> {frontend_colors.get(label, '#cccccc')}")

if "BAND_COLORS" in globals():
    backend_only = sorted(set(BAND_COLORS.keys()) - set(frontend_colors.keys()))
    frontend_only = sorted(set(frontend_colors.keys()) - set(BAND_COLORS.keys()))
    print("\nBackend vs frontend label differences:")
    print(f"  backend-only labels: {backend_only}")
    print(f"  frontend-only labels: {frontend_only}")

Saved frontend color scale image: data/processed/ui_layers/heatmap_color_scale_frontend.png

Frontend colors used:
     <15 min -> #e8f5d0
   15-30 min -> #b8e186
   30-45 min -> #7fbc41
   45-60 min -> #fddc6c
   60-75 min -> #fdae61
   75-90 min -> #f46d43
     >90 min -> #d73027
